# Stage 1 - Retrieval-augmented caption generation (Qwen2-VL-2B)

Builds a per-label FAISS index over CLIP image embeddings, retrieves the nearest same-label training images for each target image, aggregates their `concepts` keywords, and conditions Qwen2-VL-2B on that context to write a visual caption for every train/val/test image.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION - edit this cell only.
#
# These notebooks were developed in Google Colab with the dataset on Google
# Drive, so every path below defaults to a mounted-Drive layout
# (/content/drive/MyDrive/...). Nothing else in the notebook hardcodes a path.
# To run elsewhere, either set the DERM_* environment variables or edit the
# fallback strings, and skip the drive.mount() cell.
# ---------------------------------------------------------------------------
import os
from pathlib import Path

DRIVE_ROOT    = os.environ.get("DERM_DRIVE_ROOT", "/content/drive/MyDrive")
WORK_DIR      = os.environ.get("DERM_WORK_DIR", "/content")

DATASETS_ROOT = f"{DRIVE_ROOT}/Skin_Concepts/Skin_Concepts_datasets"
DATA_ROOT     = f"{DATASETS_ROOT}/fitzpatrick17k/data"   # train/val/test CSVs + image folders
BASE          = f"{DATA_ROOT}/finalfitz17k"              # alt. layout used by some cells
CAPTIONS_DIR  = f"{DATASETS_ROOT}/captions_qwen_rag"     # Stage-1 caption CSVs
RUNS_DIR      = f"{DRIVE_ROOT}/SmolVLM_runs"             # checkpoints / logs / eval CSVs
EVAL_DIR      = f"{DRIVE_ROOT}/Fitz"                     # held-out eval predictions + references

# Stage-1 inputs
TRAIN_CSV     = f"{DATA_ROOT}/train.csv"     # columns: image_id, label, concepts
VALID_CSV     = f"{DATA_ROOT}/valid.csv"
TEST_CSV      = f"{DATA_ROOT}/test.csv"
DESC_CSV      = f"{DATA_ROOT}/description.csv"
VAL_CSV       = VALID_CSV                    # alias used by some cells

TRAIN_IMG_DIR = f"{DATA_ROOT}/Train-image"
VALID_IMG_DIR = f"{DATA_ROOT}/Valid-image"
TEST_IMG_DIR  = f"{DATA_ROOT}/Test-image"
BASE_IMG_PATH = DATA_ROOT                    # root holding the three image folders

# Stage-1 outputs
OUT_DIR       = CAPTIONS_DIR
CACHE_DIR     = f"{DATA_ROOT}/faiss_cache"   # FAISS index + metadata cache
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Models / retrieval knobs
MODEL_ID    = "Qwen/Qwen2-VL-2B-Instruct"
EMB_MODEL   = "clip-ViT-L-14"   # CLIP image encoder used to build the FAISS index
SCRUB_DIAG  = False             # True -> strip diagnosis words from context/cues/output
K_NEIGHBORS = 12                # visual neighbours retrieved per image (within-label)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DATA_ROOT   :", DATA_ROOT)
print("CAPTIONS_DIR:", CAPTIONS_DIR)


In [ ]:
from google.colab import drive
drive.mount(f'{WORK_DIR}/drive')

Mounted at /content/drive


In [ ]:
# Paths and knobs now live in the CONFIGURATION cell at the top.
# (This cell previously redefined six Drive paths inline.)


Path helpers

In [ ]:
# If test.csv already has 'image_path' column, it will be used. Otherwise we infer {image_id}.{ext}
def resolve_test_path(row):
    if "image_path" in row and isinstance(row["image_path"], str) and Path(row["image_path"]).exists():
        return row["image_path"]
    iid = str(row["image_id"])
    for ext in [".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"]:
        p = Path(TEST_IMG_DIR) / f"{iid}{ext}"
        if p.exists(): return str(p)
    raise FileNotFoundError(f"Test image not found for image_id={iid}")

def resolve_train_path(iid):
    iid = str(iid)
    for ext in [".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"]:
        p = Path(TRAIN_IMG_DIR) / f"{iid}{ext}"
        if p.exists(): return str(p)
    return None  # we’ll skip missing ones when building the visual index


Build label-level text context from the training `concepts` column (optionally joined with a per-label description CSV)

In [ ]:
def clean_text(x):
    x = str(x or "")
    return re.sub(r"\s+", " ", x).strip()

agg = (
    train.groupby("label")["concepts"]
         .apply(lambda s: " ; ".join(sorted(set([str(x) for x in s if isinstance(x, str)]))))
         .reset_index(name="concepts_agg")
)

label_ctx = desc[["label","description"]].merge(agg, on="label", how="left")
label_ctx["description"]  = label_ctx["description"].map(clean_text)
label_ctx["concepts_agg"] = label_ctx["concepts_agg"].map(clean_text)

def make_label_context(row, max_chars=1200):
    parts = []
    if row["description"]:
        parts.append(f"Description: {row['description']}")
    if row["concepts_agg"]:
        parts.append(f"Keywords: {row['concepts_agg']}")
    return "\n".join(parts)[:max_chars]

label_ctx["label_context"] = label_ctx.apply(make_label_context, axis=1)
label2ctx = dict(zip(label_ctx["label"], label_ctx["label_context"]))


Visual RAG: embed training images & build FAISS index (CLIP)

In [ ]:
# --- replace your FAISS per-label build block with this ---
label_index = {}

def build_label_index(paths, ids):
    # embed
    embs = embed_images(paths, batch=32)                  # numpy float32, normalized
    embs = np.ascontiguousarray(embs.astype(np.float32))  # ensure C-contiguous
    d = int(embs.shape[1])                                # <- ensure plain Python int

    # inner product (cosine if embeddings are L2-normalized)
    idx = faiss.IndexFlatIP(d)
    idx.add(embs)
    return idx, ids

for lb, g in tqdm(train_subset.groupby("label"), desc="FAISS per-label"):
    paths = g["path"].tolist()
    ids   = g["image_id"].tolist()
    if not paths:
        continue
    try:
        idx, ids_kept = build_label_index(paths, ids)
        label_index[lb] = (idx, ids_kept)
    finally:
        gc.collect()


FAISS per-label: 100%|██████████| 114/114 [2:44:05<00:00, 86.37s/it]


Build per-image visual neighbor keywords (within the same label)

In [ ]:
# Map image_id -> tokenized concepts from train
import re

def split_keywords(s):
    s = str(s or "")
    toks = re.split(r"[;,/|]", s)
    return [t.strip().lower() for t in toks if t.strip()]

img2concepts = dict(zip(train["image_id"], train["concepts"].map(split_keywords)))

# Diagnosis words to suppress in prompts
BAN_DIAG = {"melanoma","carcinoma","sarcoma","cancer","malignant","benign","metastatic"}

def top_k_visual_keywords(test_img_path, label, k_nn=12, max_terms=12):
    """Find top-k visually similar train images within the SAME label,
       collect frequent non-diagnostic keywords."""
    if label not in label_index:
        return []
    idx, ids = label_index[label]
    q = embed_images([test_img_path], batch=1)  # uses your existing embed_images
    D, I = idx.search(q, min(k_nn, len(ids)))
    picked = [ids[i] for i in I[0] if 0 <= i < len(ids)]
    counts = {}
    for tid in picked:
        for kw in img2concepts.get(tid, []):
            if kw in BAN_DIAG or len(kw) < 3:
                continue
            counts[kw] = counts.get(kw, 0) + 1
    items = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    return [w for w,_ in items[:max_terms]]


In [ ]:
SYSTEM_PROMPT = (
    "You are a medical imaging assistant. Describe ONLY what is visible in the image.\n"
    "- The image is authoritative; text context is optional background.\n"
    "- If a finding is uncertain or not visible, do NOT state it.\n"
    "- Prefer modality terms (dermoscopic, clinical photo; radiograph/CT/MRI/US if applicable).\n"
    "- Write ONE concise clinical sentence. Avoid diagnosis terms."
)

def make_user_prompt(label_context:str, visual_terms:list):
    parts = []
    if visual_terms:
        parts.append("Likely visual cues from similar images: " + ", ".join(sorted(set(visual_terms))) + ".")
    if label_context:
        parts.append(label_context[:800])   # keep short
    if parts:
        return ("Use the background below only if it clearly matches the image.\n" +
                "\n".join(parts) +
                "\nTask: Describe the image in ONE factual sentence based on visible evidence.")
    else:
        return "Task: Describe the image in ONE factual sentence based on visible evidence."


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"  # or "Qwen/Qwen2.5-VL-3B-Instruct"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
    # Uncomment if you run out of VRAM (needs bitsandbytes):
    # load_in_4bit=True,
)

def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": [
            {"type":"text","text": user_prompt},
            {"type":"image","image": image}
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    pix = processor.image_processor(image, return_tensors="pt").pixel_values.to(model.device)

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs, pixel_values=pix,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )
    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    if "assistant" in text:
        text = text.split("assistant")[-1].strip().lstrip(":").strip()
    return text


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
# MUST use the placeholder + images=[image] pattern for Qwen-VL
def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": [
            {"type": "image"},                         # placeholder only
            {"type": "text", "text": user_prompt}
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        images=[image],             # <-- pass the PIL image here
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )
    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    if "assistant" in text:
        text = text.split("assistant")[-1].strip().lstrip(":").strip()
    return text


In [ ]:
def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    # Both roles use list-style content; image is a placeholder here.
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": system_prompt}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},                      # placeholder only
                {"type": "text", "text": user_prompt},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        images=[image],                 # <-- pass the PIL image here
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )

    text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # strip any leading role label added by the template
    if "assistant" in text:
        text = text.split("assistant")[-1].strip().lstrip(":").strip()
    return text


In [ ]:
def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    messages = [
        {"role": "system", "content": [{"type":"text","text": system_prompt}]},
        {"role": "user",   "content": [{"type":"text","text": user_prompt}]},  # no image token in the template
    ]
    # Render template to a single text string (no image token needed here)
    rendered = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,   # <-- get plain text
    )

    # Now pack text + image together
    model_inputs = processor(
        text=[rendered],
        images=[image],
        return_tensors="pt"
    )
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    with torch.inference_mode():
        out_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )

    return processor.batch_decode(out_ids, skip_special_tokens=True)[0].strip()


In [ ]:
def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    # 1) Build messages WITH an image placeholder
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user",   "content": [
            {"type": "image"},                              # <-- placeholder
            {"type": "text", "text": user_prompt},
        ]},
    ]

    # 2) Render to text so the template inserts the image tokens
    rendered = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,              # get plain text that includes image tokens
    )

    # 3) Pack BOTH text and the actual PIL image
    model_inputs = processor(
        text=[rendered],
        images=[image],
        return_tensors="pt",
    )
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    # 4) Generate
    with torch.inference_mode():
        out_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )

    return processor.batch_decode(out_ids, skip_special_tokens=True)[0].strip()


In [ ]:
from PIL import Image
from tqdm import tqdm
import re, os, pandas as pd

test10 = test.head(10).copy()
has_label = "label" in test10.columns
records = []

for _, row in tqdm(test10.iterrows(), total=len(test10), desc="Captioning-10"):
    try:
        img_path = resolve_test_path(row)
    except FileNotFoundError:
        continue

    label = str(row["label"]) if has_label else None
    label_context = label2ctx.get(label, "") if label else ""
    visual_terms  = top_k_visual_keywords(img_path, label, k_nn=10, max_terms=10) if label else []

    user_with_rag = make_user_prompt(label_context, visual_terms)
    user_no_rag   = "Task: Describe the image in ONE factual sentence based on visible evidence."

    with Image.open(img_path).convert("RGB") as im:
        cap_A = generate_caption(im, SYSTEM_PROMPT, user_with_rag, max_new_tokens=80, do_sample=False)
        cap_B = generate_caption(im, SYSTEM_PROMPT, user_no_rag,   max_new_tokens=80, do_sample=False)

    diag_terms = {"melanoma","carcinoma","sarcoma","cancer","malignant","benign","metastatic"}
    def risky(c): return bool(set(re.findall(r"[a-zA-Z]+", c.lower())) & diag_terms)
    final_cap = cap_B if (len(cap_A) > len(cap_B) + 40 or risky(cap_A)) else cap_A

    rec = {
        "image_id": row["image_id"],
        "caption": final_cap,
        "caption_with_rag": cap_A,
        "caption_no_rag": cap_B,
        "used_visual_terms": ", ".join(visual_terms),
        "image_path": img_path
    }
    if has_label: rec["label"] = label
    records.append(rec)

pred10 = pd.DataFrame(records)
out10 = os.path.join(OUT_DIR, "predictions_qwen_rag_first10.csv")
pred10.to_csv(out10, index=False)
print("Saved ->", out10)
display(pred10[["image_id"] + (["label"] if has_label else []) + ["caption","used_visual_terms"]])


Captioning-10: 100%|██████████| 10/10 [00:30<00:00,  3.02s/it]

Saved -> /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/outputs/predictions_qwen_rag_first10.csv


                           image_id                    label  \
0  d395430d11d4ac72e6f60360aabf0e61       pustular psoriasis   
1  8dffbb994ef17963995f8059a76418b9             folliculitis   
2  01be7f7454385c1abaa9d10aabcaa751            fordyce spots   
3  e1e0b7f3462e4d9c5819e81c22d8238d           scleromyxedema   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a                  scabies   
5  101fb0b2fc67b2abc246c005e22bf9fc                porphyria   
6  e7c328dc6222fcfc044c10008c28b272       pustular psoriasis   
7  d10f7d686f7bea14ccba5de443458a56  squamous cell carcinoma   
8  fc446fa1b75e9e020388e015cf288b8b            drug eruption   
9  587758cb98eb5433866465fcbe6d032b    neurotic excoriations   

                                             caption  \
0  system\nYou are a medical imaging assistant. D...   
1  system\nYou are a medical imaging assistant. D...   
2  system\nYou are a medical imaging assistant. D...   
3  system\nYou are a medical imaging assistant. D...   
4  system\nYou 

In [ ]:
import re

def _clean_chat_output(text: str) -> str:
    """
    Remove chat-template prefixes like 'system ... user ... assistant' and markers.
    Keep only the assistant's final message.
    """
    s = text.strip()

    # 1) Prefer split on the last 'assistant' marker if present
    #    (covers plain 'assistant', '<|im_start|>assistant', 'ASSISTANT:', etc.)
    markers = [r"<\|im_start\|>\s*assistant", r"\bassistant\b", r"ASSISTANT\s*:?", r"Assistant\s*:?"]
    last_pos = -1
    for pat in markers:
        for m in re.finditer(pat, s, flags=re.IGNORECASE):
            last_pos = max(last_pos, m.end())
    if last_pos != -1:
        s = s[last_pos:].lstrip(":").strip()

    # 2) Strip any trailing end markers if the template inserted them
    s = re.sub(r"<\|im_end\|>\s*$", "", s).strip()

    # 3) Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def generate_caption(image, system_prompt, user_prompt, max_new_tokens=96, do_sample=False):
    # Build messages WITH an image placeholder
    messages = [
        {"role": "system", "content": [{"type":"text","text": system_prompt}]},
        {"role": "user",   "content": [
            {"type":"image"},
            {"type":"text","text": user_prompt},
        ]},
    ]

    # Render to text so image tokens are inserted, then pack text+image
    rendered = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    model_inputs = processor(text=[rendered], images=[image], return_tensors="pt")
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    with torch.inference_mode():
        out_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=None,
            num_beams=1,
            repetition_penalty=1.05,
        )

    raw = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    return _clean_chat_output(raw)


In [ ]:
pred10["caption"] = pred10["caption"].map(_clean_chat_output)
pred10["caption_with_rag"] = pred10["caption_with_rag"].map(_clean_chat_output)
pred10["caption_no_rag"] = pred10["caption_no_rag"].map(_clean_chat_output)
display(pred10[["image_id","label","caption","used_visual_terms"]])


                           image_id                    label  \
0  d395430d11d4ac72e6f60360aabf0e61       pustular psoriasis   
1  8dffbb994ef17963995f8059a76418b9             folliculitis   
2  01be7f7454385c1abaa9d10aabcaa751            fordyce spots   
3  e1e0b7f3462e4d9c5819e81c22d8238d           scleromyxedema   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a                  scabies   
5  101fb0b2fc67b2abc246c005e22bf9fc                porphyria   
6  e7c328dc6222fcfc044c10008c28b272       pustular psoriasis   
7  d10f7d686f7bea14ccba5de443458a56  squamous cell carcinoma   
8  fc446fa1b75e9e020388e015cf288b8b            drug eruption   
9  587758cb98eb5433866465fcbe6d032b    neurotic excoriations   

                                             caption  \
0  The image shows a person with a rash that appe...   
1  The image shows a skin lesion with a central a...   
2  The image shows a close-up of a foot with a fo...   
3  The image shows a close-up of skin with a redd...   
4  The image sh

In [ ]:
# ==== PIPELINE EXPLAINER FOR ONE TEST IMAGE ====
# Requirements: variables & functions already defined earlier in your notebook:
# train, test, label2ctx, label_index, img2concepts, resolve_test_path,
# embed_images, top_k_visual_keywords, SYSTEM_PROMPT, make_user_prompt, generate_caption

import json, re
from pprint import pprint
from PIL import Image

# 1) pick a row to explain (change idx to any test row you want)
idx = 0
row = test.iloc[idx]

ex = {}
ex["image_id"] = str(row["image_id"])
ex["label"]    = str(row["label"]) if "label" in test.columns else None

# 2) resolve the test image path
img_path = resolve_test_path(row)
ex["image_path"] = img_path

# 3) get label-level text context (description + aggregated concepts)
label_context = label2ctx.get(ex["label"], "") if ex["label"] else ""
ex["label_context_preview"] = (label_context[:500] + "…") if len(label_context) > 500 else label_context

# 4) visual RAG: nearest neighbors within SAME label → frequent non-diagnostic terms
visual_terms = top_k_visual_keywords(img_path, ex["label"], k_nn=12, max_terms=12) if ex["label"] else []
ex["visual_terms"] = visual_terms

# (optional) show the actual neighbor image_ids and their keywords for transparency
if ex["label"] in label_index:
    faiss_idx, ids = label_index[ex["label"]]
    q = embed_images([img_path], batch=1)
    D, I = faiss_idx.search(q, min(12, len(ids)))
    nn_ids = [ids[i] for i in I[0] if 0 <= i < len(ids)]
    ex["nearest_neighbor_image_ids"] = nn_ids
    ex["nearest_neighbor_keywords"]  = {iid: img2concepts.get(iid, []) for iid in nn_ids}

# 5) build the two prompts (with and without RAG)
user_with_rag = make_user_prompt(label_context, visual_terms)
user_no_rag   = "Task: Describe the image in ONE factual sentence based on visible evidence."
ex["system_prompt"] = SYSTEM_PROMPT
ex["user_prompt_with_rag"] = user_with_rag
ex["user_prompt_no_rag"]   = user_no_rag

# 6) run Qwen-VL twice (A: with RAG, B: image-only) and pick the safer caption
with Image.open(img_path).convert("RGB") as im:
    cap_A = generate_caption(im, SYSTEM_PROMPT, user_with_rag, max_new_tokens=80, do_sample=False)
    cap_B = generate_caption(im, SYSTEM_PROMPT, user_no_rag,   max_new_tokens=80, do_sample=False)

diag_terms = {"melanoma","carcinoma","sarcoma","cancer","malignant","benign","metastatic"}
def risky(c): return bool(set(re.findall(r"[a-zA-Z]+", c.lower())) & diag_terms)
final_cap = cap_B if (len(cap_A) > len(cap_B) + 40 or risky(cap_A)) else cap_A

ex["caption_with_rag"] = cap_A
ex["caption_no_rag"]   = cap_B
ex["final_caption"]    = final_cap
ex["chooser_reason"]   = (
    "Used NO-RAG (safer)" if final_cap == cap_B else "Used RAG (concise & safe)"
)

# 7) pretty print summary
print("=== EXPLANATION FOR TEST INDEX", idx, "===")
print("Image ID  :", ex["image_id"])
print("Label     :", ex["label"])
print("Image Path:", ex["image_path"])
print("\n-- Visual RAG terms (from nearest neighbors) --")
print(", ".join(ex["visual_terms"]) or "(none)")
print("\n-- Label text context (preview) --")
print(ex["label_context_preview"] or "(none)")
print("\n-- User prompt WITH RAG (shown to Qwen) --\n", ex["user_prompt_with_rag"])
print("\n-- User prompt NO RAG --\n", ex["user_prompt_no_rag"])
print("\n-- Generated (WITH RAG) --\n", ex["caption_with_rag"])
print("\n-- Generated (NO RAG) --\n", ex["caption_no_rag"])
print("\n===> FINAL CAPTION:", ex["final_caption"], f"  [{ex['chooser_reason']}]")

# 8) save a JSON artifact next to your outputs (for auditing)
import os
os.makedirs(OUT_DIR, exist_ok=True)
dbg_path = os.path.join(OUT_DIR, f"explain_{ex['image_id']}.json")
with open(dbg_path, "w") as f:
    json.dump(ex, f, indent=2)
print("\nSaved explanation JSON ->", dbg_path)


=== EXPLANATION FOR TEST INDEX 0 ===
Image ID  : d395430d11d4ac72e6f60360aabf0e61
Label     : pustular psoriasis
Image Path: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/Test-image/d395430d11d4ac72e6f60360aabf0e61.jpg

-- Visual RAG terms (from nearest neighbors) --
'erythematous skin', 'painful skin', 'pustular psoriasis', 'recurrent flares', 'superficial pustules.'], 'systemic inflammation', 'von zumbusch psoriasis', 'widespread sterile pustules', ['generalised pustular psoriasis'

-- Label text context (preview) --
Description: Generalised pustular psoriasis (GPP) is a rare, severe form of pustular psoriasis . It is characterised by recurrent flares of widespread sterile pustules with erythematous , painful skin. GPP can be associated with systemic inflammation including fevers and/or hepatic , gastrointestinal, musculoskeletal , renal , or pulmonary involvement. It is also known as von Zumbusch psoriasis. Generalised pustular psoriasis Generalised

In [ ]:
# ===== PATCH: make final captions actually use your train data (RAG) =====
import re
import pandas as pd
from PIL import Image

# 1) Rebuild a diagnosis-neutral context per label
DIAG_BAN = {"melanoma","carcinoma","sarcoma","cancer","malignant","benign","metastatic"}

def neutralize_diagnoses(text: str, label: str) -> str:
    s = str(text or "")
    # remove label tokens + banned diagnosis words
    label_tokens = [t for t in re.split(r"[^a-zA-Z]+", str(label or "")) if t]
    ban = set(map(str.lower, label_tokens)) | DIAG_BAN
    for w in sorted(ban, key=len, reverse=True):
        s = re.sub(rf"\b{re.escape(w)}\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# assumes label_ctx with columns ["label","label_context"] already exists from earlier steps
label_ctx["label_context_neutral"] = label_ctx.apply(
    lambda r: neutralize_diagnoses(r["label_context"], r["label"]), axis=1
)
label2ctx = dict(zip(label_ctx["label"], label_ctx["label_context_neutral"]))  # <-- use neutral context

# 2) Prompt that *encourages* using 1–3 cues when visible
SYSTEM_PROMPT = (
    "You are a medical imaging assistant. Describe ONLY what is visible in the image.\n"
    "- The image is authoritative; text context is optional background.\n"
    "- If a finding is uncertain or not visible, do NOT state it.\n"
    "- Prefer modality terms (dermoscopic, clinical photo; radiograph/CT/MRI/US if applicable).\n"
    "- Write ONE concise clinical sentence. Avoid diagnosis terms."
)

def make_user_prompt(label_context:str, visual_terms:list):
    # keep short; ask to use 1–3 cues if they visibly apply
    vt = ", ".join(sorted(set(visual_terms[:10])))
    parts = []
    if vt:
        parts.append(f"Likely visual cues from similar images: {vt}.")
        parts.append("When appropriate and clearly visible, incorporate 1–3 of the cues above into the sentence.")
    if label_context:
        parts.append(label_context[:600])
    if parts:
        return ("Use the background below only if it clearly matches the image.\n" +
                "\n".join(parts) +
                "\nTask: Describe the image in ONE factual sentence based on visible evidence.")
    else:
        return "Task: Describe the image in ONE factual sentence based on visible evidence."

# 3) Simple “influence” score: how much of the RAG cues made it into the caption?
def influence_score(caption: str, visual_terms: list) -> int:
    cap = caption.lower()
    score = 0
    for t in set(visual_terms):
        t = t.lower().strip()
        if len(t) < 3:
            continue
        if re.search(rf"\b{re.escape(t)}\b", cap):
            score += 1
    return score

def risky(caption: str) -> bool:
    toks = set(re.findall(r"[a-zA-Z]+", caption.lower()))
    return bool(toks & DIAG_BAN)

# 4) Pilot on first 10 — prefer WITH-RAG if it uses cues and isn’t risky
test10 = test.head(10).copy()
has_label = "label" in test10.columns
rows = []

for _, row in test10.iterrows():
    # resolve image path (your resolve_test_path from earlier)
    try:
        img_path = resolve_test_path(row)
    except FileNotFoundError:
        continue

    label = str(row["label"]) if has_label else None
    label_context = label2ctx.get(label, "") if label else ""
    visual_terms  = top_k_visual_keywords(img_path, label, k_nn=12, max_terms=12) if label else []

    user_with_rag = make_user_prompt(label_context, visual_terms)
    user_no_rag   = "Task: Describe the image in ONE factual sentence based on visible evidence."

    with Image.open(img_path).convert("RGB") as im:
        cap_A = generate_caption(im, SYSTEM_PROMPT, user_with_rag, max_new_tokens=80, do_sample=False)  # WITH RAG
        cap_B = generate_caption(im, SYSTEM_PROMPT, user_no_rag,   max_new_tokens=80, do_sample=False)  # NO RAG

    # decision: use RAG if it (a) isn't risky and (b) shows influence (>=1 cue)
    infl = influence_score(cap_A, visual_terms)
    if risky(cap_A):
        final_cap, picker = cap_B, "no-rag (rag looked diagnostic)"
    elif infl >= 1:
        final_cap, picker = cap_A, f"rag (influence={infl})"
    elif len(cap_A) <= len(cap_B) + 20:
        final_cap, picker = cap_A, "rag (concise tie-break)"
    else:
        final_cap, picker = cap_B, "no-rag (weak influence)"

    rows.append({
        "image_id": row["image_id"],
        "label": label,
        "caption": final_cap,
        "caption_with_rag": cap_A,
        "caption_no_rag": cap_B,
        "used_visual_terms": ", ".join(visual_terms),
        "rag_influence_score": infl,
        "picker_reason": picker
    })

pred10 = pd.DataFrame(rows)
from IPython.display import display
display(pred10[["image_id","label","caption","rag_influence_score","picker_reason","used_visual_terms"]])


                           image_id                    label  \
0  d395430d11d4ac72e6f60360aabf0e61       pustular psoriasis   
1  8dffbb994ef17963995f8059a76418b9             folliculitis   
2  01be7f7454385c1abaa9d10aabcaa751            fordyce spots   
3  e1e0b7f3462e4d9c5819e81c22d8238d           scleromyxedema   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a                  scabies   
5  101fb0b2fc67b2abc246c005e22bf9fc                porphyria   
6  e7c328dc6222fcfc044c10008c28b272       pustular psoriasis   
7  d10f7d686f7bea14ccba5de443458a56  squamous cell carcinoma   
8  fc446fa1b75e9e020388e015cf288b8b            drug eruption   
9  587758cb98eb5433866465fcbe6d032b    neurotic excoriations   

                                             caption  rag_influence_score  \
0  A patient with generalized pustular psoriasis ...                    0   
1  The image shows a tender red spot with a surfa...                    0   
2  The image shows a close-up of a foot with visi...            

Caption with RAG

In [ ]:
# ===== First-10 captions USING RAG (always) =====
import re, os, pandas as pd
from PIL import Image
from IPython.display import display

# 1) Scrub diagnosis words from VISUAL terms (keeps descriptive cues only)
BASE_DIAG_BAN = {
    "melanoma","carcinoma","sarcoma","cancer","malignant","benign","metastatic",
    "psoriasis","eczema","dermatitis","folliculitis","acne","impetigo","cellulitis",
    "rosacea","tinea","dermatophytosis","scabies","urticaria","lichen","pemphigus",
    "pemphigoid","vitiligo","melasma","keratosis","nevus","naevus","ulcer","mycosis",
    "hidradenitis","suppurativa","warts","verruca","molluscum","herpes","zoster",
    "lupus","scleroderma","sarcoidosis","dermatomyositis","actinic","keratoses"
}

def label_tokens(label: str):
    return [t for t in re.split(r"[^a-zA-Z]+", str(label or "")) if t]

def clean_terms(terms, label):
    ban = set(t.lower() for t in label_tokens(label)) | BASE_DIAG_BAN
    out = []
    for w in terms:
        wl = str(w).lower().strip()
        if not wl or any(re.search(rf"\b{re.escape(b)}\b", wl) for b in ban):
            continue
        if len(wl) >= 3:
            out.append(wl)
    # de-dup and keep up to 12 cues
    out = sorted(set(out))[:12]
    return out

# 2) Prompt helper (explicitly ask to include 1–3 cues when visible)
def make_user_prompt_rag(label_context:str, visual_terms:list):
    vt = ", ".join(visual_terms[:10])
    parts = []
    if vt:
        parts.append(f"Likely visual cues from similar images: {vt}.")
        parts.append("If clearly visible, explicitly include 1–3 of the cues above in your ONE-sentence description.")
    if label_context:
        parts.append(label_context[:600])  # concise background
    body = "\n".join(parts)
    if body:
        return ("Use the background below only if it matches the image.\n" +
                body + "\nTask: Describe the image in ONE factual sentence based on visible evidence.")
    return "Task: Describe the image in ONE factual sentence based on visible evidence."

# 3) Run on the first 10 test rows — ALWAYS use the RAG prompt
rows = []
test1000 = test.head(1000).copy()
has_label = "label" in test1000.columns

for _, row in test1000.iterrows():
    # resolve image path
    try:
        img_path = resolve_test_path(row)
    except Exception:
        continue

    label = str(row["label"]) if has_label else ""
    # label_context should already be neutralized in your label2ctx; if not, it still works
    label_context = label2ctx.get(label, "")

    # visual RAG (neighbors) → cleaned descriptive terms
    terms_raw = top_k_visual_keywords(img_path, label, k_nn=12, max_terms=24) if label else []
    visual_terms = clean_terms(terms_raw, label)

    user_prompt = make_user_prompt_rag(label_context, visual_terms)

    with Image.open(img_path).convert("RGB") as im:
        caption_rag = generate_caption(
            im, SYSTEM_PROMPT, user_prompt,
            max_new_tokens=80, do_sample=False
        )

    rows.append({
        "image_id": row["image_id"],
        "label": label,
        "caption": caption_rag,                 # ← produced WITH RAG
        "used_visual_terms": ", ".join(visual_terms),
        "image_path": img_path
    })

pred1000_rag = pd.DataFrame(rows)
display(pred1000_rag[["image_id","label","caption","used_visual_terms"]])

# (optional) save
out1000 = os.path.join(OUT_DIR, "predictions_qwen_WITH_RAG_first10.csv")
pred1000_rag.to_csv(out10, index=False)
print("Saved ->", out10)


                             image_id               label  \
0    d395430d11d4ac72e6f60360aabf0e61  pustular psoriasis   
1    8dffbb994ef17963995f8059a76418b9        folliculitis   
2    01be7f7454385c1abaa9d10aabcaa751       fordyce spots   
3    e1e0b7f3462e4d9c5819e81c22d8238d      scleromyxedema   
4    c7fcb5f49fbe7fb7eeec7eaf196b299a             scabies   
..                                ...                 ...   
995  73e337bc592e0756d47339af2bc4076a     photodermatoses   
996  245748cdc16cd75c81615f233f69d3fe           psoriasis   
997  e0c06db3391d6dd80d3f6f27f11e5416         scleroderma   
998  211086e05c40844993bd42a0c6f5e806              eczema   
999  9b6fe93d271142f4f29eb9d117276110           psoriasis   

                                               caption  \
0    A patient with generalized pustular psoriasis ...   
1    An inflamed hair follicle with a tender red sp...   
2    The image shows a close-up of a foot with visi...   
3    Lichen myxoedematosus is a rar

Saved -> /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/outputs/predictions_qwen_WITH_RAG_first10.csv


In [ ]:
!pip -q install --upgrade transformers accelerate pillow timm einops sentencepiece
!pip -q install sentence-transformers faiss-cpu bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 44.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pillow<12.0,>=8.0, but you have pillow 12.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 28.5 MB/s eta 0:00:00


In [ ]:
import os

BASE = BASE

# Count all image files (jpg, jpeg, png, etc.)
img_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
count = sum(
    len([f for f in files if f.lower().endswith(img_exts)])
    for _, _, files in os.walk(BASE)
)

print(f"Total images in '{BASE}': {count}")


In [ ]:
import os, re, gc, json, glob, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch, faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModelForVision2Seq
from google.colab import drive
drive.mount(f'{WORK_DIR}/drive')

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# =======================
def must_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing: raise ValueError(f"{name} missing columns: {missing}")

train = pd.read_csv(TRAIN_CSV)
valid = pd.read_csv(VALID_CSV)
test  = pd.read_csv(TEST_CSV)
desc  = pd.read_csv(DESC_CSV)

must_cols(train, ["image_id","label","concepts"], "train.csv")
must_cols(valid, ["image_id","label"], "valid.csv")
must_cols(test,  ["image_id","label"], "test.csv")
must_cols(desc,  ["label","description"], "description.csv")

# =======================
# PROPAGATE concepts from TRAIN -> VALID/TEST by label
# =======================
def split_keywords(s):
    return [t.strip() for t in re.split(r"[;,/|]", str(s or "")) if t.strip()]

def join_keywords(items):
    seen, out = set(), []
    for x in items:
        if x not in seen:
            seen.add(x); out.append(x)
    return " ; ".join(out)

label2concepts = (
    train.assign(_kw=train["concepts"].map(split_keywords))
         .explode("_kw")
         .dropna(subset=["_kw"])
         .groupby("label")["_kw"]
         .apply(lambda s: join_keywords(list(s)))
         .reset_index(name="concepts")
)

valid = valid.merge(label2concepts, on="label", how="left")
test  = test.merge(label2concepts,  on="label", how="left")


In [ ]:
# If any label in valid/test doesn’t exist in train (rare), fallback to simple keywords from description
missing_valid = valid["concepts"].isna().sum()
missing_test  = test["concepts"].isna().sum()
if missing_valid or missing_test:
    STOP = set("the a an and or of for to with in on by from as is are was were be being been that this these those it its at into about over under within without may can often usually typically common commonly such including".split())
    def desc_to_keywords(text, k=15):
        words = re.findall(r"[A-Za-z]{3,}", str(text or "").lower())
        words = [w for w in words if w not in STOP]
        freq = {}
        for w in words: freq[w] = freq.get(w, 0) + 1
        return " ; ".join([w for w,_ in sorted(freq.items(), key=lambda x:(-x[1], x[0]))[:k]])
    fallback = desc.assign(concepts=desc["description"].map(desc_to_keywords))[["label","concepts"]]
    if missing_valid:
        valid = valid.drop(columns=["concepts"]).merge(fallback, on="label", how="left")
    if missing_test:
        test  = test.drop(columns=["concepts"]).merge(fallback, on="label",  how="left")

train["split"] = "train"
valid["split"] = "valid"
test["split"]  = "test"
all_df = pd.concat([train, valid, test], ignore_index=True)

print("Rows per split:\n", all_df["split"].value_counts())
print("Unique labels:", all_df["label"].nunique())

# =======================
# Label-level text RAG (description + agg concepts)
# =======================
def clean_text(x): return re.sub(r"\s+", " ", str(x or "")).strip()
agg_concepts = (
    all_df.groupby("label")["concepts"]
          .apply(lambda s: " ; ".join(sorted(set([str(x) for x in s if isinstance(x, str)]))))
          .reset_index(name="concepts_agg")
)
label_ctx = desc[["label","description"]].merge(agg_concepts, on="label", how="left")
label_ctx["description"]  = label_ctx["description"].map(clean_text)
label_ctx["concepts_agg"] = label_ctx["concepts_agg"].map(clean_text)
def make_label_context(row, max_chars=1200):
    parts = []
    if row["description"]:  parts.append(f"Description: {row['description']}")
    if row["concepts_agg"]: parts.append(f"Keywords: {row['concepts_agg']}")
    return "\n".join(parts)[:max_chars]
label_ctx["label_context"] = label_ctx.apply(make_label_context, axis=1)
label2ctx = dict(zip(label_ctx["label"], label_ctx["label_context"]))
print("label2ctx ready for", len(label2ctx), "labels")


Rows per split:
 split
train    11787
test      3316
valid     1474
Name: count, dtype: int64
Unique labels: 114
label2ctx ready for 114 labels


In [ ]:
!pip -q install transformers accelerate pillow tqdm --upgrade
import os, sys, json, time, math, gc
from pathlib import Path

import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm

from transformers import AutoProcessor, AutoModelForCausalLM  # Qwen2-VL uses CausalLM + Processor


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.4 MB/s eta 0:00:00


In [ ]:
def resolve_path(image_id, split):
    subdir_map = {
        "train": "Train-image",
        "val":   "Valid-image",
        "valid": "Valid-image",
        "test":  "Test-image"
    }
    subdir = subdir_map.get(split, "Train-image")
    for ext in [".jpg", ".jpeg", ".png"]:
        path = f"{BASE_IMG_PATH}/{subdir}/{image_id}{ext}"
        if os.path.exists(path):
            return path
    return None



In [ ]:
test_id = train.iloc[0]["image_id"]
for s in ["train", "val", "test"]:
    p = resolve_path(test_id, s)
    print(s, "→", p if p else "❌ not found")



train → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/Train-image/16a32b978ee680b9e7be70b56324cc2e.jpg
val → ❌ not found
test → ❌ not found


In [ ]:
def load_csvs(train_csv, val_csv, test_csv, desc_csv):
    train = pd.read_csv(train_csv)
    val   = pd.read_csv(val_csv)
    test  = pd.read_csv(test_csv)
    desc  = pd.read_csv(desc_csv)

    # Normalize column names
    train.columns = [c.strip().lower() for c in train.columns]
    val.columns   = [c.strip().lower() for c in val.columns]
    test.columns  = [c.strip().lower() for c in test.columns]
    desc.columns  = [c.strip().lower() for c in desc.columns]

    # Common expectations
    assert {"image_id", "label"}.issubset(set(train.columns))
    assert {"image_id", "label"}.issubset(set(val.columns))
    assert {"image_id", "label"}.issubset(set(test.columns))
    assert {"label", "description"}.issubset(set(desc.columns))

    # Keywords column may be named "keywords" or "concepts"; make an alias
    if "keywords" not in train.columns and "concepts" in train.columns:
        train = train.rename(columns={"concepts": "keywords"})
    if "keywords" not in train.columns:
        train["keywords"] = ""

    return train, val, test, desc

train, val, test, desc = load_csvs(TRAIN_CSV, VAL_CSV, TEST_CSV, DESC_CSV)
len(train), len(val), len(test), len(desc)


(11787, 1474, 3316, 114)

In [ ]:
# Aggregate ALL keywords in train per label (in case multiple rows per label)
kw_by_label = (
    train.groupby("label")["keywords"]
         .apply(lambda s: "; ".join(sorted(set(str(x) for x in s if pd.notna(x) and str(x).strip()))))
         .to_dict()
)

# Description lookup
desc_map = {str(r["label"]): str(r["description"]) for _, r in desc.iterrows()}

# Build final context map; fallback gracefully if keywords are missing for a label
label2ctx = {}
all_labels = set(train["label"]).union(set(val["label"])).union(set(test["label"]))
for lbl in sorted(all_labels):
    l = str(lbl)
    kw  = kw_by_label.get(l, "")
    dsc = desc_map.get(l, "")
    parts = []
    if kw.strip():  parts.append(f"Keywords: {kw}")
    if dsc.strip(): parts.append(f"Description: {dsc}")
    ctx = " ".join(parts).strip()
    # ultimate fallback: at least echo label
    if not ctx:
        ctx = f"Label: {l}"
    label2ctx[l] = ctx

print(f"Contexts ready for {len(label2ctx)} labels.")
# Peek a couple
for i, (k,v) in enumerate(label2ctx.items()):
    if i>=3: break
    print("—", k, "→", (v[:180] + "..." if len(v)>180 else v))


Contexts ready for 114 labels.
— acanthosis nigricans → Keywords: ['Acanthosis nigricans', 'velvety', 'papillomatous overgrowth', 'epidermis', 'darkening', 'thickening', 'epidermal hyperplasia', 'skin', 'axillary acanthosis nigricans'] ...
— acne → Keywords: ['Acne', 'chronic disorder', 'affecting hair follicle', 'sebaceous gland', 'expansion of follicle', 'blockage of follicle', 'inflammation.'] Description: Acne is a common...
— acne vulgaris → Keywords: ['Acne', 'inflammatory papules', 'pustules', 'nodules', 'non-inflamed comedones', 'pseudocysts', 'oily skin', 'seborrhoea', 'postinflammatory hyperpigmentation', 'pigment...


In [ ]:
!pip -q uninstall -y transformers
!pip -q install "transformers>=4.45.0" "accelerate>=0.33.0" "qwen-vl-utils>=0.0.8" sentencepiece --upgrade
import torch, pandas as pd, os, gc


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 60.8 MB/s eta 0:00:00


In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # or "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are a careful dermatology assistant. "
    "Use the provided context as medical hints, but describe only what the image supports. "
    "Be precise, neutral, and avoid hallucinations."
)

def build_prompt(label_context: str):
    # You can tune this wording
    return (
        f"<image>\n"
        f"Context: {label_context}\n\n"
        f"Task: Write a clinically useful caption for this skin lesion image. "
        f"Include morphology, distribution, color/shape (if visible), and likely diagnosis cues. "
        f"If uncertain, say so briefly."
    )


In [ ]:
def resolve_path(image_id: str, split: str):
    subdir = {
        "train": "Train-image",
        "val":   "Valid-image",
        "valid": "Valid-image",
        "test":  "Test-image"
    }[split]
    # Change extension if yours are .png or mixed
    p_jpg = Path(BASE_IMG_DIR) / subdir / f"{image_id}.jpg"
    p_png = Path(BASE_IMG_DIR) / subdir / f"{image_id}.png"
    if p_jpg.exists():
        return str(p_jpg)
    if p_png.exists():
        return str(p_png)
    return None


In [ ]:
@torch.inference_mode()
def generate_caption_for_image(img_path: str, label: str, ctx: str,
                               max_new_tokens=128, temperature=0.2, top_p=0.9):
    if img_path is None or not Path(img_path).exists():
        return "", "MISSING_IMAGE"

    try:
        image = Image.open(img_path).convert("RGB")
        user_prompt = build_prompt(ctx)
        inputs = processor(
            text=[{"role": "system", "content": SYSTEM_INSTRUCTION},
                  {"role": "user",   "content": user_prompt}],
            images=[image],
            return_tensors="pt"
        )
        for k in inputs:
            inputs[k] = inputs[k].to(model.device, dtype=torch.float16 if inputs[k].dtype==torch.float else None)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )
        # decode
        text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        return text, "OK"
    except Exception as e:
        return "", f"ERROR: {e}"


In [ ]:
def caption_split_resumable(df: pd.DataFrame, split: str, out_csv: str):
    # Normalize columns
    df = df.copy()
    df["image_id"] = df["image_id"].astype(str)
    df["label"]    = df["label"].astype(str)

    # Resume: load already done image_ids
    done = set()
    if Path(out_csv).exists():
        prev = pd.read_csv(out_csv)
        if "image_id" in prev.columns:
            done = set(prev["image_id"].astype(str))
        print(f"[{split}] Resuming — already has {len(done)} rows")

    # Stream append
    out_cols = ["image_id", "label", "context", "caption", "status", "image_path"]
    out_fh = open(out_csv, "a", encoding="utf-8")
    if Path(out_csv).stat().st_size == 0:
        out_fh.write(",".join(out_cols) + "\n")  # header

    new_count = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Generating ({split})"):
        iid = str(r["image_id"])
        if iid in done:
            continue

        lbl = str(r["label"])
        ctx = label2ctx.get(lbl, f"Label: {lbl}")  # safety fallback
        ip  = resolve_path(iid, split)

        caption, status = generate_caption_for_image(ip, lbl, ctx)
        row = [
            iid.replace(",", " "),
            lbl.replace(",", " "),
            (ctx.replace("\n", " ").replace(",", "; ")),
            (caption.replace("\n", " ").replace(",", "; ")),
            status,
            (ip or "")
        ]
        out_fh.write(",".join(row) + "\n")
        new_count += 1

        # occasional flush & GC for long runs
        if new_count % 100 == 0:
            out_fh.flush()
            gc.collect()

    out_fh.close()
    print(f"[{split}] Wrote {new_count} new rows → {out_csv}")


In [ ]:
from pathlib import Path
import pandas as pd, gc

def caption_split_resumable(df: pd.DataFrame, split: str, out_csv: str):
    df = df.copy()
    df["image_id"] = df["image_id"].astype(str)
    df["label"]    = df["label"].astype(str)

    done = set()
    p = Path(out_csv)
    if p.exists() and p.stat().st_size > 0:
        try:
            prev = pd.read_csv(out_csv)
            if "image_id" in prev.columns:
                done = set(prev["image_id"].astype(str))
            print(f"[{split}] Resuming — already has {len(done)} rows")
        except Exception as e:
            print(f"[{split}] Warning: could not read existing CSV ({e}). Starting fresh.")
    else:
        if p.exists():
            print(f"[{split}] Found empty file → will write header fresh.")
        else:
            print(f"[{split}] No previous file → starting fresh.")

    out_cols = ["image_id", "label", "context", "caption", "status", "image_path"]
    mode = "a"
    fh = open(out_csv, mode, encoding="utf-8")
    # Write header if file is new or empty
    if (not p.exists()) or (p.stat().st_size == 0):
        fh.write(",".join(out_cols) + "\n")

    new_count = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Generating ({split})"):
        iid = str(r["image_id"])
        if iid in done:
            continue
        lbl = str(r["label"])
        ctx = label2ctx.get(lbl, f"Label: {lbl}")
        ip  = resolve_path(iid, split)

        cap, status = generate_caption_for_image(ip, lbl, ctx)
        row = [
            iid.replace(",", " "),
            lbl.replace(",", " "),
            (ctx.replace("\n"," ").replace(",", "; ")),
            (cap.replace("\n"," ").replace(",", "; ")),
            status,
            (ip or "")
        ]
        fh.write(",".join(row) + "\n")
        new_count += 1
        if new_count % 100 == 0:
            fh.flush(); gc.collect()
    fh.close()
    print(f"[{split}] Wrote {new_count} new rows → {out_csv}")


In [ ]:
!pip -q uninstall -y transformers -y
!pip -q install "transformers>=4.45.0" "accelerate>=0.33.0" "qwen-vl-utils>=0.0.8" sentencepiece --upgrade

import os, gc, json
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 25.7 MB/s eta 0:00:00


In [ ]:
# Base folder that contains Train-image / Valid-image / Test-image and CSVs
BASE_IMG_PATH = DATA_ROOT

# Where to write captions
OUT_DIR = CAPTIONS_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# Helper: pick first existing path from a list (for val filename variants)
def first_existing(*paths):
    for p in paths:
        if p and Path(p).exists():
            return p
    return None

TRAIN_CSV = first_existing(f"{BASE_IMG_PATH}/train.csv", f"{WORK_DIR}/train.csv")
VAL_CSV   = first_existing(f"{BASE_IMG_PATH}/val.csv",
                           f"{BASE_IMG_PATH}/val - val.csv.csv",
                           f"{WORK_DIR}/val - val.csv.csv", f"{WORK_DIR}/val.csv")
TEST_CSV  = first_existing(f"{BASE_IMG_PATH}/test.csv", f"{WORK_DIR}/test - test.csv.csv", f"{WORK_DIR}/test.csv")
DESC_CSV  = first_existing(f"{BASE_IMG_PATH}/description.csv", f"{WORK_DIR}/description.csv")

print("CSV files:")
print("  train:", TRAIN_CSV)
print("  val  :", VAL_CSV)
print("  test :", TEST_CSV)
print("  desc :", DESC_CSV)


CSV files:
  train: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/train.csv
  val  : /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/val.csv
  test : /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/test.csv
  desc : /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/description.csv


In [ ]:
def load_csvs(train_csv, val_csv, test_csv, desc_csv):
    assert train_csv and Path(train_csv).exists(), "train.csv not found"
    assert test_csv  and Path(test_csv ).exists(), "test.csv not found"
    assert desc_csv  and Path(desc_csv ).exists(), "description.csv not found"
    if not (val_csv and Path(val_csv).exists()):
        print("⚠️ val CSV not found; using empty DataFrame")
        val = pd.DataFrame(columns=["image_id","label"])
    else:
        val = pd.read_csv(val_csv)

    train = pd.read_csv(train_csv)
    test  = pd.read_csv(test_csv)
    desc  = pd.read_csv(desc_csv)

    for df in (train, val, test, desc):
        df.columns = [c.strip().lower() for c in df.columns]

    # map concepts->keywords if needed
    if "keywords" not in train.columns and "concepts" in train.columns:
        train = train.rename(columns={"concepts":"keywords"})
    if "keywords" not in train.columns:
        train["keywords"] = ""

    assert {"image_id","label"}.issubset(train.columns)
    assert {"image_id","label"}.issubset(test.columns)
    if not val.empty:
        assert {"image_id","label"}.issubset(val.columns)
    assert {"label","description"}.issubset(desc.columns)

    return train, val, test, desc

train, val, test, desc = load_csvs(TRAIN_CSV, VAL_CSV, TEST_CSV, DESC_CSV)
print(len(train), len(val), len(test), len(desc))


11787 1474 3316 114


In [ ]:
# aggregate all keywords per label from train
kw_by_label = (
    train.groupby("label")["keywords"]
         .apply(lambda s: "; ".join(sorted({str(x).strip() for x in s if pd.notna(x) and str(x).strip()})))
         .to_dict()
)
desc_map = {str(r["label"]): str(r["description"]) for _, r in desc.iterrows()}

all_labels = set(train["label"]).union(set(val["label"])).union(set(test["label"]))
label2ctx = {}
for lbl in sorted(all_labels):
    l = str(lbl)
    parts = []
    if kw_by_label.get(l, "").strip():
        parts.append(f"Keywords: {kw_by_label[l]}")
    if desc_map.get(l, "").strip():
        parts.append(f"Description: {desc_map[l]}")
    ctx = " ".join(parts).strip() or f"Label: {l}"
    label2ctx[l] = ctx

print(f"Contexts built for {len(label2ctx)} labels.")


Contexts built for 114 labels.


In [ ]:
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # or "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are a careful dermatology assistant. "
    "Use the provided context as hints, but only describe what the image supports. "
    "Be precise, neutral, and avoid hallucinations."
)

def build_prompt(ctx: str) -> str:
    return (
        f"Context: {ctx}\n\n"
        f"Task: Write a clinically useful caption for this skin lesion image. "
        f"Include morphology, distribution, color/shape (if visible), and likely diagnosis cues. "
        f"If uncertain, say so briefly."
    )

def resolve_path(image_id: str, split: str):
    subdir = {"train":"Train-image", "val":"Valid-image", "valid":"Valid-image", "test":"Test-image"}.get(split,"Train-image")
    for ext in (".jpg",".jpeg",".png",".tif",".tiff",".bmp"):
        p = Path(BASE_IMG_PATH) / subdir / f"{image_id}{ext}"
        if p.exists():
            return str(p)
    return None

@torch.inference_mode()
def generate_caption_for_image(img_path: str, label: str, ctx: str,
                               max_new_tokens=128, temperature=0.2, top_p=0.9):
    if not img_path or not Path(img_path).exists():
        return "", "MISSING_IMAGE"
    try:
        image = Image.open(img_path).convert("RGB")
        user_text = build_prompt(ctx)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCTION}]},
            {"role":"user","content":[
                {"type":"image","image":image},
                {"type":"text","text":user_text}
            ]},
        ]

        input_ids = processor.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
        image_inputs = processor.image_processor([image], return_tensors="pt")

        inputs = {"input_ids": input_ids, **image_inputs}
        inputs = {k:v.to(model.device) for k,v in inputs.items()}

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
        )
        text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        return text, "OK"
    except Exception as e:
        return "", f"ERROR: {e}"


In [ ]:
def caption_split_resumable(df: pd.DataFrame, split: str, out_csv: str):
    df = df.copy()
    if df.empty:
        print(f"[{split}] Input is empty; skipping.")
        return
    df["image_id"] = df["image_id"].astype(str)
    df["label"]    = df["label"].astype(str)

    done = set()
    p = Path(out_csv)
    if p.exists() and p.stat().st_size > 0:
        try:
            prev = pd.read_csv(out_csv)
            if "image_id" in prev.columns:
                done = set(prev["image_id"].astype(str))
            print(f"[{split}] Resuming — already has {len(done)} rows")
        except Exception as e:
            print(f"[{split}] Warning: could not read existing CSV ({e}). Starting fresh.")
    else:
        if p.exists():
            print(f"[{split}] Found empty file → will write header fresh.")
        else:
            print(f"[{split}] No previous file → starting fresh.")

    out_cols = ["image_id","label","context","caption","status","image_path"]
    fh = open(out_csv, "a", encoding="utf-8")
    if (not p.exists()) or (p.stat().st_size == 0):
        fh.write(",".join(out_cols) + "\n")

    new_count = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Generating ({split})"):
        iid = str(r["image_id"])
        if iid in done:
            continue
        lbl = str(r["label"])
        ctx = label2ctx.get(lbl, f"Label: {lbl}")
        ip  = resolve_path(iid, split)

        cap, status = generate_caption_for_image(ip, lbl, ctx)
        # sanitize commas/newlines for CSV
        write_row = [
            iid.replace(",", " "),
            lbl.replace(",", " "),
            (ctx or "").replace("\n"," ").replace(",", "; "),
            (cap or "").replace("\n"," ").replace(",", "; "),
            status,
            ip or ""
        ]
        fh.write(",".join(write_row) + "\n")
        new_count += 1
        if new_count % 100 == 0:
            fh.flush(); gc.collect()

    fh.close()
    print(f"[{split}] Wrote {new_count} new rows → {out_csv}")


In [ ]:
probe_id = train.iloc[0]["image_id"]
for s in ["train","val","test"]:
    p = resolve_path(probe_id, s)
    print(f"{s:<5} →", p if p else "❌ not found (ok if this ID isn't in that split)")


train → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/fitzpatrick17k/data/Train-image/16a32b978ee680b9e7be70b56324cc2e.jpg
val   → ❌ not found (ok if this ID isn't in that split)
test  → ❌ not found (ok if this ID isn't in that split)


In [ ]:
@torch.inference_mode()
def generate_caption_for_image(img_path: str, label: str, ctx: str,
                               max_new_tokens=128, temperature=0.2, top_p=0.9):
    if not img_path or not Path(img_path).exists():
        return "", "MISSING_IMAGE"
    try:
        image = Image.open(img_path).convert("RGB")
        # inject RAG context straight into the prompt
        prompt = (
            "You are a careful dermatology assistant. "
            "Use the provided context as hints, but only describe what the image supports. "
            "Be precise, neutral, and avoid hallucinations.\n\n"
            f"Context: {ctx}\n\n"
            "Task: Write a clinically useful caption for this skin lesion image. "
            "Include morphology, distribution, color/shape (if visible), and likely diagnosis cues. "
            "If uncertain, say so briefly."
        )

        # <<< KEY: single-call processor creates a clean tensor batch >>>
        inputs = processor(text=prompt, images=image, return_tensors="pt")
        inputs = inputs.to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
        )
        text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        return text, "OK"
    except Exception as e:
        return "", f"ERROR: {e}"


FINAL

In [ ]:
# ===== Visual-only captioning: prompt, sanitizer, caption fn, resumable writer, preview =====
import os, re, gc, textwrap
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm

# ---- If model/processor not loaded yet, load a reasonable default ----
try:
    processor
    model
except NameError:
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
    MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # or "Qwen/Qwen2.5-VL-3B-Instruct"
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )

# ---- Inference knobs (lower randomness = fewer speculations) ----
CAP_MAX_NEW_TOKENS = 90
CAP_TEMPERATURE    = 0.1
CAP_TOP_P          = 0.8

# ---- STRICT visual-only system instruction + prompt builder ----
SYSTEM_INSTRUCTION = (
    "You are a vision-only describer for dermatology photos. STRICT RULES:\n"
    "• Describe ONLY what is visible in the image (morphology, color, shape, size, borders, distribution, location, surface/texture).\n"
    "• DO NOT give diagnoses, probabilities, differential diagnoses, or disease names.\n"
    "• DO NOT give management, treatment, prognosis, or recommendations.\n"
    "• Use neutral, concise, observational language. No speculation."
)

def build_prompt(ctx: str) -> str:
    # Context is just vocabulary hints; not for diagnosis/treatment
    return (
        "Visual description task.\n"
        "Use the context as vocabulary hints only; do NOT infer diagnoses.\n"
        f"Context (hints): {ctx}\n\n"
        "Write 1–3 neutral sentences that describe the visible findings: "
        "lesion type(s), color(s), size/scale if inferable, border/shape, distribution/location, surface features "
        "(e.g., scaling, crust, ulceration), and notable surrounding skin changes. "
        "Do NOT mention disease names, likelihoods, or any treatment."
    )

# ---- Post-filter to remove any accidental advice/diagnosis sentences ----
_BANNED = re.compile(
    r"(treat|treatment|therapy|medicat|topical|oral|surgery|excision|biopsy|"
    r"recommend|follow[- ]?up|manage|management|prognos|monitor|"
    r"diagnos|likely|suggests|consistent with|indicative of|differential)",
    re.IGNORECASE
)

def sanitize_caption(text: str) -> str:
    sents = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    kept  = [s for s in sents if s and not _BANNED.search(s)]
    if not kept and sents:           # fallback: keep shortest purely visual sentence
        kept = [min(sents, key=len)]
    cleaned = re.sub(r"\s{2,}", " ", " ".join(kept)).strip()
    return cleaned

# ---- Image resolver (uses your existing folder layout) ----
def resolve_path(image_id: str, split: str):
    subdir = {"train":"Train-image", "val":"Valid-image", "valid":"Valid-image", "test":"Test-image"}.get(split,"Train-image")
    for ext in (".jpg",".jpeg",".png",".tif",".tiff",".bmp"):
        p = Path(BASE_IMG_PATH) / subdir / f"{image_id}{ext}"
        if p.exists():
            return str(p)
    return None

# ---- Core captioner: correct text↔image alignment + decode only generated tokens ----
@torch.inference_mode()
def generate_caption_for_image(img_path: str, label: str, ctx: str,
                               max_new_tokens=CAP_MAX_NEW_TOKENS, temperature=CAP_TEMPERATURE, top_p=CAP_TOP_P):
    if not img_path or not Path(img_path).exists():
        return "", "MISSING_IMAGE"
    try:
        image = Image.open(img_path).convert("RGB")

        user_text = build_prompt(ctx)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
            {"role": "user",   "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": user_text},
            ]},
        ]

        # 1) Build chat string with image token placeholders (no tokenization yet)
        chat_str = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )

        # 2) Pack text+image into one aligned batch
        batch = processor(text=[chat_str], images=[image], return_tensors="pt").to(model.device)

        # 3) Generate
        outputs = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
        )

        # 4) Decode ONLY the newly generated tokens
        input_len = batch["input_ids"].shape[-1]
        gen_ids   = outputs[:, input_len:]
        text_out  = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

        # 5) Visual-only cleanup
        text_out = sanitize_caption(text_out)
        return text_out, "OK"

    except Exception as e:
        return "", f"ERROR: {e}"

# ---- Resume-safe writer (now with sanitizer) ----
def caption_split_resumable(df: pd.DataFrame, split: str, out_csv: str):
    df = df.copy()
    if df.empty:
        print(f"[{split}] Input is empty; skipping.")
        return
    df["image_id"] = df["image_id"].astype(str)
    df["label"]    = df["label"].astype(str)

    done = set()
    p = Path(out_csv)
    if p.exists() and p.stat().st_size > 0:
        try:
            prev = pd.read_csv(out_csv)
            if "image_id" in prev.columns:
                done = set(prev["image_id"].astype(str))
            print(f"[{split}] Resuming — already has {len(done)} rows")
        except Exception as e:
            print(f"[{split}] Warning: could not read existing CSV ({e}). Starting fresh.")
    else:
        print(f"[{split}] No previous file → starting fresh.")

    out_cols = ["image_id","label","context","caption","status","image_path"]
    fh = open(out_csv, "a", encoding="utf-8")
    if (not p.exists()) or (p.stat().st_size == 0):
        fh.write(",".join(out_cols) + "\n")

    new_count = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Generating ({split})"):
        iid = str(r["image_id"])
        if iid in done:
            continue

        lbl = str(r["label"])
        ctx = label2ctx.get(lbl, f"Label: {lbl}") if 'label2ctx' in globals() else f"Label: {lbl}"
        ip  = resolve_path(iid, split)

        cap, status = generate_caption_for_image(ip, lbl, ctx)
        if status == "OK":
            cap = sanitize_caption(cap)

        # CSV-safe
        row = [
            iid.replace(",", " "),
            lbl.replace(",", " "),
            (ctx or "").replace("\n"," ").replace(",", "; "),
            (cap or "").replace("\n"," ").replace(",", "; "),
            status,
            ip or ""
        ]
        fh.write(",".join(row) + "\n")
        new_count += 1
        if new_count % 100 == 0:
            fh.flush(); gc.collect()

    fh.close()
    print(f"[{split}] Wrote {new_count} new rows → {out_csv}")

# ---- Preview helper: show image + caption inline ----
import matplotlib.pyplot as plt
def show_examples(csv_path: str, split="train", n=5):
    df = pd.read_csv(csv_path)
    rows = df.head(n)
    for _, r in rows.iterrows():
        iid   = str(r["image_id"])
        label = str(r["label"])
        cap   = str(r.get("caption","")).strip()
        ipath = resolve_path(iid, split)
        print(f"\nID: {iid} | Label: {label}")
        if not ipath:
            print("❌ Image not found")
            continue
        img = Image.open(ipath).convert("RGB")
        plt.figure(figsize=(6,6))
        plt.imshow(img); plt.axis("off")
        plt.title(textwrap.fill(cap if cap else "(no caption)", width=90), fontsize=10, loc="left")
        plt.show()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
sample_out = f"{OUT_DIR}/captions_train_sample10_visual_only.csv"
from pathlib import Path
if Path(sample_out).exists(): Path(sample_out).unlink()
caption_split_resumable(train.head(10), "train", sample_out)
show_examples(sample_out, split="train", n=5)


[train] No previous file → starting fresh.


Generating (train): 100%|██████████| 10/10 [00:42<00:00,  4.27s/it]


[train] Wrote 10 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/captions_train_sample10_visual_only.csv

ID: 16a32b978ee680b9e7be70b56324cc2e | Label: hidradenitis


<Figure size 600x600 with 1 Axes>


ID: 71a101b8e494eb213b78a1274714de3b | Label: keloid


...

<Figure size 600x600 with 1 Axes>


ID: be22e0c423de5d2529b57efb5bae7327 | Label: behcets disease


<Figure size 600x600 with 1 Axes>


ID: 4ebb78e463dd08523ced087cb11b1650 | Label: syringoma


<Figure size 600x600 with 1 Axes>

In [ ]:
# ===== Run full captioning on TRAIN / VAL / TEST (resume-safe) =====
from pathlib import Path
import pandas as pd, os

# Output files
train_out = f"{OUT_DIR}/againnewcaptions_train_qwen_rag_visual_only.csv"
val_out   = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"
test_out  = f"{OUT_DIR}/againnewcaptions_test_qwen_rag_visual_only.csv"

# Clean up any 0-byte leftovers from earlier
for f in (train_out, val_out, test_out):
    if Path(f).exists() and Path(f).stat().st_size == 0:
        Path(f).unlink()
        print("Removed empty file:", f)

# ---- Run (resume-safe; skips already processed image_ids) ----
caption_split_resumable(train, "train", train_out)
caption_split_resumable(val,   "val",   val_out)
caption_split_resumable(test,  "test",  test_out)

# ---- Quick summary ----
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok  = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss= (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(train_out, "train")
summarize(val_out,   "val")
summarize(test_out,  "test")

# ---- Optional: export minimal CSVs (image_id, label, caption only) ----
for full, mini in [
    (train_out, f"{OUT_DIR}/newcaptions_train_minimal.csv"),
    (val_out,   f"{OUT_DIR}/newcaptions_val_minimal.csv"),
    (test_out,  f"{OUT_DIR}/newcaptions_test_minimal.csv"),
]:
    if Path(full).exists():
        df = pd.read_csv(full)
        cols = [c for c in ["image_id","label","caption"] if c in df.columns]
        pd.DataFrame(df[cols]).to_csv(mini, index=False)

# ---- Optional: preview a few from each split inline ----
print("\nPreview TRAIN:")
show_examples(train_out, split="train", n=3)

print("\nPreview VAL:")
show_examples(val_out, split="val", n=3)

print("\nPreview TEST:")
show_examples(test_out, split="test", n=3)


[train] Resuming — already has 11787 rows


Generating (train): 100%|██████████| 11787/11787 [00:00<00:00, 28211.94it/s]


[train] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_train_qwen_rag_visual_only.csv
[val] Resuming — already has 1474 rows


Generating (val): 100%|██████████| 1474/1474 [00:00<00:00, 27235.50it/s]


[val] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
[test] Resuming — already has 3316 rows


Generating (test): 100%|██████████| 3316/3316 [00:00<00:00, 28166.79it/s]


[test] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_test_qwen_rag_visual_only.csv
TRAIN → rows: 11787 | OK: 0 | MISSING_IMAGE: 11787 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_train_qwen_rag_visual_only.csv
VAL   → rows:  1474 | OK: 0 | MISSING_IMAGE: 1474 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
TEST  → rows:  3316 | OK: 0 | MISSING_IMAGE: 3316 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/captions_qwen_rag/againnewcaptions_test_qwen_rag_visual_only.csv

Preview TRAIN:

ID: 16a32b978ee680b9e7be70b56324cc2e | Label: hidradenitis
❌ Image not found

ID: 71a101b8e494eb213b78a1274714de3b | Label: keloid
❌ Image not found

ID: 5b9ca4868b5919ead2a0ff0ac520d014 | Label: lichen planus
❌ Image not found

Preview VAL:

ID: 436d32d92e72a91eeeb0351870a0ca83 | Label: acquired au

In [ ]:
# ===== Run full captioning on TRAIN / VAL / TEST (resume-safe, first 5k) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (Colab disconnect prevention)
try:
    from google.colab import output
    # JS keep-alive ping every 60s to prevent browser idle timeout
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    # Kernel-level heartbeat to prevent backend idle disconnect
    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output files
train_out = f"{OUT_DIR}/newcaptions_train_first5k_qwen_rag_visual_only.csv"
val_out   = f"{OUT_DIR}/anewcaptions_val_qwen_rag_visual_only.csv"
test_out  = f"{OUT_DIR}/nnewcaptions_test_qwen_rag_visual_only.csv"

# Remove empty CSVs if they exist
for f in (train_out, val_out, test_out):
    if Path(f).exists() and Path(f).stat().st_size == 0:
        Path(f).unlink()
        print("Removed empty file:", f)

# -------------------------------------------------------------------
# 🧠 Run captioning (resume-safe; skips already processed image_ids)
# Limit training set to first 5000 rows
train_subset = train.head(5000).reset_index(drop=True)
print(f"Running only on the first {len(train_subset)} samples from TRAIN split.")

caption_split_resumable(train_subset, "train", train_out)
caption_split_resumable(val, "val", val_out)
caption_split_resumable(test, "test", test_out)

# -------------------------------------------------------------------
# 📊 Summary for each split
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(train_out, "train")
summarize(val_out, "val")
summarize(test_out, "test")

# -------------------------------------------------------------------
# 💾 Export minimal CSVs (image_id, label, caption only)
for full, mini in [
    (train_out, f"{OUT_DIR}/newcaptions_train_first5k_minimal.csv"),
    (val_out,   f"{OUT_DIR}/newcaptions_val_minimal.csv"),
    (test_out,  f"{OUT_DIR}/newcaptions_test_minimal.csv"),
]:
    if Path(full).exists():
        df = pd.read_csv(full)
        cols = [c for c in ["image_id","label","caption"] if c in df.columns]
        pd.DataFrame(df[cols]).to_csv(mini, index=False)
        print("Wrote:", mini)

# -------------------------------------------------------------------
# 🖼️ Optional: Preview examples
print("\nPreview TRAIN:")
show_examples(train_out, split="train", n=3)

print("\nPreview VAL:")
show_examples(val_out, split="val", n=3)

print("\nPreview TEST:")
show_examples(test_out, split="test", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Running only on the first 5000 samples from TRAIN split.
[train] No previous file → starting fresh.


Generating (train):   0%|          | 13/5000 [00:42<5:09:41,  3.73s/it]

.

Generating (train):   0%|          | 17/5000 [00:58<5:31:00,  3.99s/it]

.

...

Generating (test):  14%|█▍        | 472/3316 [32:15<3:12:24,  4.06s/it]

.

Generating (test):  14%|█▍        | 475/3316 [32:28<3:14:08,  4.10s/it]

.

Generating (test):  14%|█▍        | 476/3316 [32:32<3:13:00,  4.08s/it]

In [ ]:
# ===== Run captioning on next 5k (rows 5001–10,000) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (prevent Colab disconnect)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output files for this chunk
train_out = f"{OUT_DIR}/againnewcaptions_train_5kto10k_qwen_rag_visual_only.csv"
val_out   = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"
test_out  = f"{OUT_DIR}/againnewcaptions_test_qwen_rag_visual_only.csv"

# Remove empty files if any
for f in (train_out, val_out, test_out):
    if Path(f).exists() and Path(f).stat().st_size == 0:
        Path(f).unlink()
        print("Removed empty file:", f)

# -------------------------------------------------------------------
# 🧠 Use only rows 5001–10,000 from train
train_subset = train.iloc[5000:10000].reset_index(drop=True)
print(f"Processing next {len(train_subset)} samples (rows 5001–10,000).")

# Run resume-safe captioning
caption_split_resumable(train_subset, "train", train_out)
caption_split_resumable(val, "val", val_out)
caption_split_resumable(test, "test", test_out)

# -------------------------------------------------------------------
# 📊 Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(train_out, "train")
summarize(val_out, "val")
summarize(test_out, "test")

# -------------------------------------------------------------------
# 💾 Minimal CSV exports
for full, mini in [
    (train_out, f"{OUT_DIR}/newcaptions_train_5kto10k_minimal.csv"),
    (val_out,   f"{OUT_DIR}/newcaptions_val_minimal.csv"),
    (test_out,  f"{OUT_DIR}/newcaptions_test_minimal.csv"),
]:
    if Path(full).exists():
        df = pd.read_csv(full)
        cols = [c for c in ["image_id","label","caption"] if c in df.columns]
        pd.DataFrame(df[cols]).to_csv(mini, index=False)
        print("Wrote:", mini)

# -------------------------------------------------------------------
# 🖼️ Optional preview
print("\nPreview TRAIN:")
show_examples(train_out, split="train", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Removed empty file: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
Processing next 5000 samples (rows 5001–10,000).
[train] No previous file → starting fresh.


Generating (train):   0%|          | 15/5000 [00:57<5:30:17,  3.98s/it]

.

Generating (train):   1%|          | 30/5000 [01:56<5:47:46,  4.20s/it]

.

...

Generating (test):  23%|██▎       | 771/3316 [50:15<2:40:36,  3.79s/it]

.

Generating (test):  24%|██▎       | 785/3316 [51:16<3:05:59,  4.41s/it]

.

Generating (test):  24%|██▍       | 790/3316 [51:34<2:41:48,  3.84s/it]

In [ ]:
# ===== Run captioning on next 10k batch (rows 10,001–20,000) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (prevent Colab disconnect)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output files for this range
train_out = f"{OUT_DIR}/againnewcaptions_train_10kto20k_qwen_rag_visual_only.csv"
val_out   = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"
test_out  = f"{OUT_DIR}/againnewcaptions_test_qwen_rag_visual_only.csv"

# Remove 0-byte files if any
for f in (train_out, val_out, test_out):
    if Path(f).exists() and Path(f).stat().st_size == 0:
        Path(f).unlink()
        print("Removed empty file:", f)

# -------------------------------------------------------------------
# 🧠 Use only rows 10,001–20,000 from train
train_subset = train.iloc[10000:20000].reset_index(drop=True)
print(f"Processing next {len(train_subset)} samples (rows 10,001–20,000).")

# Run captioning (resume-safe)
caption_split_resumable(train_subset, "train", train_out)
caption_split_resumable(val, "val", val_out)
caption_split_resumable(test, "test", test_out)

# -------------------------------------------------------------------
# 📊 Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(train_out, "train")
summarize(val_out, "val")
summarize(test_out, "test")

# -------------------------------------------------------------------
# 💾 Minimal CSV export (image_id, label, caption only)
for full, mini in [
    (train_out, f"{OUT_DIR}/newcaptions_train_10kto20k_minimal.csv"),
    (val_out,   f"{OUT_DIR}/newcaptions_val_minimal.csv"),
    (test_out,  f"{OUT_DIR}/newcaptions_test_minimal.csv"),
]:
    if Path(full).exists():
        df = pd.read_csv(full)
        cols = [c for c in ["image_id","label","caption"] if c in df.columns]
        pd.DataFrame(df[cols]).to_csv(mini, index=False)
        print("Wrote:", mini)

# -------------------------------------------------------------------
# 🖼️ Optional preview
print("\nPreview TRAIN:")
show_examples(train_out, split="train", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Processing next 1787 samples (rows 10,001–20,000).
[train] No previous file → starting fresh.


Generating (train):   1%|          | 18/1787 [00:59<1:35:20,  3.23s/it]

.

Generating (train):   2%|▏         | 36/1787 [01:57<1:34:38,  3.24s/it]

.

...

<Figure size 600x600 with 1 Axes>


ID: 09206ca265fec54fe9b43059c981b660 | Label: tuberous sclerosis


<Figure size 600x600 with 1 Axes>


ID: 8f0044b44f57159eb1ae7c6dbc7b9e76 | Label: allergic contact dermatitis


<Figure size 600x600 with 1 Axes>

In [ ]:
# ===== Run full captioning on VALIDATION split (resume-safe) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (prevent Colab from disconnecting)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output files
val_out = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"
test_out = f"{OUT_DIR}/againnewcaptions_test_qwen_rag_visual_only.csv"

# Clean up any 0-byte files
for f in (val_out, test_out):
    if Path(f).exists() and Path(f).stat().st_size == 0:
        Path(f).unlink()
        print("Removed empty file:", f)

# -------------------------------------------------------------------
# 🧠 Process the entire validation split
print(f"Processing full validation set — total {len(val)} samples.")
caption_split_resumable(val, "val", val_out)
caption_split_resumable(test, "test", test_out)

# -------------------------------------------------------------------
# 📊 Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(val_out, "val")
summarize(test_out, "test")

# -------------------------------------------------------------------
# 💾 Save a minimal version (image_id, label, caption only)
for full, mini in [
    (val_out,  f"{OUT_DIR}/newcaptions_val_minimal.csv"),
    (test_out, f"{OUT_DIR}/newcaptions_test_minimal.csv"),
]:
    if Path(full).exists():
        df = pd.read_csv(full)
        cols = [c for c in ["image_id", "label", "caption"] if c in df.columns]
        pd.DataFrame(df[cols]).to_csv(mini, index=False)
        print("Wrote:", mini)

# -------------------------------------------------------------------
# 🖼️ Optional preview
print("\nPreview VAL:")
show_examples(val_out, split="val", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Processing full validation set — total 1474 samples.
[val] Resuming — already has 1474 rows


Generating (val): 100%|██████████| 1474/1474 [00:00<00:00, 27768.61it/s]

[val] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv


[test] Resuming — already has 3316 rows


Generating (test): 100%|██████████| 3316/3316 [00:00<00:00, 28288.19it/s]

...

<Figure size 600x600 with 1 Axes>


ID: 8d7b2609f03faca293c8be2077801619 | Label: sarcoidosis


<Figure size 600x600 with 1 Axes>


ID: 093fcc49c98d314303ba07aa0d6c1348 | Label: actinic keratosis


<Figure size 600x600 with 1 Axes>

In [ ]:
# ===== Caption VALIDATION only (resume-safe) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# (Optional) Keep-alive so Colab doesn't idle
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    def _hb():
        while True:
            time.sleep(60); sys.stdout.write("."); sys.stdout.flush()
    threading.Thread(target=_hb, daemon=True).start()
    print("✅ Keep-alive enabled.")
except Exception:
    pass

# Output paths (validation only)
val_out = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"

# Remove 0-byte leftovers if any
p = Path(val_out)
if p.exists() and p.stat().st_size == 0:
    p.unlink(); print("Removed empty file:", val_out)

print(f"Processing VALID split: {len(val)} rows")
caption_split_resumable(val, "val", val_out)

# Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}"); return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(val_out, "val")

# Also write a clean minimal CSV (image_id, label, caption only)
mini_val = f"{OUT_DIR}/newcaptions_val_minimal.csv"
if Path(val_out).exists():
    df = pd.read_csv(val_out)
    keep = [c for c in ["image_id","label","caption"] if c in df.columns]
    pd.DataFrame(df[keep]).to_csv(mini_val, index=False)
    print("📝 Wrote minimal CSV:", mini_val)

# (Optional) preview a few
#print("\nPreview VAL:"); show_examples(val_out, split="val", n=3)


✅ Keep-alive enabled.
Processing VALID split: 1474 rows
[val] Resuming — already has 1474 rows


Generating (val): 100%|██████████| 1474/1474 [00:00<00:00, 24912.57it/s]

[val] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
VAL   → rows:  1474 | OK: 1474 | MISSING_IMAGE: 0 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
📝 Wrote minimal CSV: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/newcaptions_val_minimal.csv


In [ ]:
# ===== Run captioning on VALID split only (resume-safe) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (prevent Colab disconnect)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output file for VALID
val_out = f"{OUT_DIR}/againnewcaptions_val_qwen_rag_visual_only.csv"

# Remove 0-byte leftovers if any
if Path(val_out).exists() and Path(val_out).stat().st_size == 0:
    Path(val_out).unlink()
    print("Removed empty file:", val_out)

# -------------------------------------------------------------------
# 🧠 Caption VALID (entire split; resume-safe)
print(f"Processing VALID split — total {len(val)} samples.")
caption_split_resumable(val, "val", val_out)

# -------------------------------------------------------------------
# 📊 Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(val_out, "val")

# -------------------------------------------------------------------
# 💾 Minimal CSV export (image_id, label, caption only)
mini_val = f"{OUT_DIR}/newcaptions_val_minimal.csv"
if Path(val_out).exists():
    df = pd.read_csv(val_out)
    cols = [c for c in ["image_id","label","caption"] if c in df.columns]
    pd.DataFrame(df[cols]).to_csv(mini_val, index=False)
    print("Wrote:", mini_val)

# -------------------------------------------------------------------
# 🖼️ Optional preview
#print("\nPreview VAL:")
#show_examples(val_out, split="val", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Processing VALID split — total 1474 samples.
[val] Resuming — already has 1474 rows


Generating (val): 100%|██████████| 1474/1474 [00:00<00:00, 27493.48it/s]

[val] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
VAL   → rows:  1474 | OK: 1474 | MISSING_IMAGE: 0 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_val_qwen_rag_visual_only.csv
Wrote: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/newcaptions_val_minimal.csv


In [ ]:
# ===== Run captioning on TEST split (resume-safe, full) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (prevent Colab disconnect)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    print("✅ Browser keep-alive enabled.")

    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Kernel heartbeat active (no idle timeout).")
except Exception:
    print("⚠️ Keep-alive unavailable (non-Colab environment).")

# -------------------------------------------------------------------
# 📂 Output file for TEST split
test_out = f"{OUT_DIR}/againnewcaptions_test_qwen_rag_visual_only.csv"

# Remove 0-byte leftovers if any
if Path(test_out).exists() and Path(test_out).stat().st_size == 0:
    Path(test_out).unlink()
    print("Removed empty file:", test_out)

# -------------------------------------------------------------------
# 🧠 Caption TEST split (resume-safe)
print(f"Processing TEST split — total {len(test)} samples.")
caption_split_resumable(test, "test", test_out)

# -------------------------------------------------------------------
# 📊 Summary
def summarize(path, split):
    if not Path(path).exists():
        print(f"{split}: file not found -> {path}")
        return
    df = pd.read_csv(path)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

summarize(test_out, "test")

# -------------------------------------------------------------------
# 💾 Minimal CSV export (image_id, label, caption only)
mini_test = f"{OUT_DIR}/newcaptions_test_minimal.csv"
if Path(test_out).exists():
    df = pd.read_csv(test_out)
    cols = [c for c in ["image_id","label","caption"] if c in df.columns]
    pd.DataFrame(df[cols]).to_csv(mini_test, index=False)
    print("Wrote:", mini_test)

# -------------------------------------------------------------------
# 🖼️ Optional preview
#print("\nPreview TEST:")
#show_examples(test_out, split="test", n=3)


✅ Browser keep-alive enabled.
✅ Kernel heartbeat active (no idle timeout).
Processing TEST split — total 3316 samples.
[test] Resuming — already has 3316 rows


Generating (test): 100%|██████████| 3316/3316 [00:00<00:00, 27454.93it/s]

[test] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_test_qwen_rag_visual_only.csv


TEST  → rows:  3316 | OK: 791 | MISSING_IMAGE: 2525 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_test_qwen_rag_visual_only.csv
Wrote: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/newcaptions_test_minimal.csv


In [ ]:
# ===== FRESH START for VAL + TEST (KEEP TRAIN SAFE) =====
from pathlib import Path
import pandas as pd, os, sys, time, threading, shutil, glob

# ---------------- Keep-alive (Colab) ----------------
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    def _hb():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_hb, daemon=True).start()
    print("✅ Keep-alive enabled.")
except Exception:
    pass

# ---------------- New output folder ----------------
ts = time.strftime("%Y%m%d_%H%M%S")
OUT_DIR_FRESH = Path(OUT_DIR) / f"fresh_run_{ts}"
OUT_DIR_FRESH.mkdir(parents=True, exist_ok=True)

val_out  = str(OUT_DIR_FRESH / "captions_val_qwen_rag_visual_only.csv")
test_out = str(OUT_DIR_FRESH / "captions_test_qwen_rag_visual_only.csv")

print(f"\n📁 Fresh outputs (VAL+TEST only) will be saved under: {OUT_DIR_FRESH}\n")

# ---------------- Run fresh for VAL + TEST ----------------
def summarize(path, split):
    p = Path(path)
    if not p.exists():
        print(f"{split}: file not found -> {path}"); return
    df = pd.read_csv(p)
    ok   = (df["status"] == "OK").sum() if "status" in df.columns else "n/a"
    miss = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else "n/a"
    print(f"{split.upper():<5} → rows: {len(df):>5} | OK: {ok} | MISSING_IMAGE: {miss} | {path}")

# ✅ Run validation
print(f"🟡 VAL fresh run — total {len(val)} rows")
caption_split_resumable(val, "val", val_out)
summarize(val_out, "val")

# ✅ Run test
print(f"🟡 TEST fresh run — total {len(test)} rows")
caption_split_resumable(test, "test", test_out)
summarize(test_out, "test")

# ---------------- Minimal CSVs ----------------
for full, mini in [
    (val_out,  OUT_DIR_FRESH / "captions_val_minimal.csv"),
    (test_out, OUT_DIR_FRESH / "captions_test_minimal.csv"),
]:
    p = Path(full)
    if p.exists():
        df = pd.read_csv(p)
        keep = [c for c in ["image_id","label","caption"] if c in df.columns]
        pd.DataFrame(df[keep]).to_csv(mini, index=False)
        print("📝 Wrote:", mini)



✅ Keep-alive enabled.

📁 Fresh outputs (VAL+TEST only) will be saved under: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/fresh_run_20251021_174013

🟡 VAL fresh run — total 1474 rows
[val] No previous file → starting fresh.


Generating (val):   0%|          | 3/1474 [00:08<1:12:30,  2.96s/it]

.

Generating (val):   0%|          | 5/1474 [00:14<1:13:20,  3.00s/it]

.

...

.

Generating (test): 100%|█████████▉| 3313/3316 [3:39:40<00:12,  4.12s/it]

.

Generating (test): 100%|██████████| 3316/3316 [3:39:51<00:00,  3.98s/it]


[test] Wrote 3316 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/fresh_run_20251021_174013/captions_test_qwen_rag_visual_only.csv
TEST  → rows:  3316 | OK: 3316 | MISSING_IMAGE: 0 | /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/fresh_run_20251021_174013/captions_test_qwen_rag_visual_only.csv
📝 Wrote: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/fresh_run_20251021_174013/captions_val_minimal.csv
📝 Wrote: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/fresh_run_20251021_174013/captions_test_minimal.csv


In [ ]:
# === Audit all caption CSV files under OUT_DIR & check coverage ===
from pathlib import Path
import pandas as pd
import re
from collections import defaultdict, Counter

# 1) Locate ALL CSVs recursively
out_dir = Path(OUT_DIR)
csv_files = sorted([p for p in out_dir.rglob("*.csv") if p.is_file()])

if not csv_files:
    print("No CSVs found under:", out_dir)
else:
    print(f"Found {len(csv_files)} CSV files under {out_dir}")

# 2) Helper to normalize columns in any CSV
def load_csv_norm(path: Path):
    df = pd.read_csv(path, low_memory=False)
    # normalize column names
    df.columns = [c.strip().lower() for c in df.columns]
    # common renames
    rename = {
        "image id": "image_id",
        "img_id": "image_id",
        "labels": "label"
    }
    for k, v in rename.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})
    # keep a safe subset
    for need in ["image_id", "label", "caption", "status"]:
        if need not in df.columns:
            # create if missing to simplify later code
            df[need] = pd.NA
    # coerce to str where needed
    df["image_id"] = df["image_id"].astype(str)
    df["label"] = df["label"].astype(str)
    return df

# 3) Guess split from filename (fallback: unknown)
def guess_split_from_name(name: str):
    n = name.lower()
    if "train" in n: return "train"
    if re.search(r"\bval\b|valid", n): return "val"
    if "test" in n: return "test"
    return "unknown"

# 4) Collect per-file stats
rows = []
per_split_covered_ids = defaultdict(set)
global_seen_ids = set()

for p in csv_files:
    try:
        df = load_csv_norm(p)
    except Exception as e:
        rows.append({
            "file": str(p),
            "split_guess": "ERROR",
            "rows": "ERROR",
            "unique_image_id": "ERROR",
            "ok": "ERROR",
            "missing_image": "ERROR",
            "text_only": "ERROR",
            "note": f"failed to read: {e}"
        })
        continue

    split_guess = guess_split_from_name(p.name)
    n_rows = len(df)
    n_unique = df["image_id"].nunique()

    # status counts if available
    ok_cnt = (df["status"] == "OK").sum() if "status" in df.columns else pd.NA
    miss_cnt = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else pd.NA
    text_only_cnt = (df["status"] == "TEXT_ONLY").sum() if "status" in df.columns else pd.NA

    rows.append({
        "file": str(p),
        "split_guess": split_guess,
        "rows": n_rows,
        "unique_image_id": n_unique,
        "ok": ok_cnt if pd.notna(ok_cnt) else "",
        "missing_image": miss_cnt if pd.notna(miss_cnt) else "",
        "text_only": text_only_cnt if pd.notna(text_only_cnt) else "",
        "note": ""
    })

    # accumulate coverage sets by split guess
    if split_guess in ("train", "val", "test"):
        per_split_covered_ids[split_guess].update(df["image_id"].dropna().astype(str).tolist())
    else:
        # unknown files: still add into the global set; might help coverage if ids are unique anyway
        global_seen_ids.update(df["image_id"].dropna().astype(str).tolist())

# 5) Show per-file summary table
summary_df = pd.DataFrame(rows).sort_values(["split_guess", "file"]).reset_index(drop=True)
print("\n=== Per-file summary ===")
display(summary_df)

# 6) Build ground-truth sets from your original splits (assumes train/val/test DataFrames exist)
train_ids = set(train["image_id"].astype(str))
val_ids   = set(val["image_id"].astype(str))
test_ids  = set(test["image_id"].astype(str))

# If there are "unknown" caption files, include them in coverage heuristically
# (they might contain train/val/test ids; intersection will still work)
covered_train = set(per_split_covered_ids["train"]) | (global_seen_ids & train_ids)
covered_val   = set(per_split_covered_ids["val"])   | (global_seen_ids & val_ids)
covered_test  = set(per_split_covered_ids["test"])  | (global_seen_ids & test_ids)

missing_train = sorted(train_ids - covered_train)
missing_val   = sorted(val_ids   - covered_val)
missing_test  = sorted(test_ids  - covered_test)

print("\n=== Coverage summary ===")
coverage_rows = [
    {"split": "train", "total_gt": len(train_ids), "covered": len(covered_train), "missing": len(missing_train)},
    {"split": "val",   "total_gt": len(val_ids),   "covered": len(covered_val),   "missing": len(missing_val)},
    {"split": "test",  "total_gt": len(test_ids),  "covered": len(covered_test),  "missing": len(missing_test)},
]
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

# 7) Duplicates check across ALL files (per split)
def find_dups_across_files(target_ids, files):
    id_counter = Counter()
    for p in files:
        try:
            df = load_csv_norm(p)
        except Exception:
            continue
        ids = set(df["image_id"].astype(str))
        id_counter.update(ids & target_ids)
    dups = [i for i, c in id_counter.items() if c > 1]
    return dups

train_dups = find_dups_across_files(train_ids, csv_files)
val_dups   = find_dups_across_files(val_ids, csv_files)
test_dups  = find_dups_across_files(test_ids, csv_files)

print("\n=== Duplicates across CSVs (same split) ===")
print(f"Train duplicates across files: {len(train_dups)}")
print(f"Val   duplicates across files: {len(val_dups)}")
print(f"Test  duplicates across files: {len(test_dups)}")

# 8) (Optional) Save missing IDs to OUT_DIR
miss_train_path = out_dir / "audit_missing_train_image_ids.csv"
miss_val_path   = out_dir / "audit_missing_val_image_ids.csv"
miss_test_path  = out_dir / "audit_missing_test_image_ids.csv"

pd.DataFrame({"image_id": missing_train}).to_csv(miss_train_path, index=False)
pd.DataFrame({"image_id": missing_val}).to_csv(miss_val_path, index=False)
pd.DataFrame({"image_id": missing_test}).to_csv(miss_test_path, index=False)

print("\nSaved missing lists to:")
print(" -", miss_train_path)
print(" -", miss_val_path)
print(" -", miss_test_path)

# 9) Totals across ALL caption CSVs (for sanity)
total_rows_all = sum(r["rows"] for r in rows if isinstance(r["rows"], (int, float)))
total_unique_ids_all = set()
for p in csv_files:
    try:
        df = load_csv_norm(p)
        total_unique_ids_all.update(df["image_id"].astype(str))
    except Exception:
        pass

print("\n=== Totals across ALL caption CSVs (not de-duplicated by split) ===")
print("Total rows (sum of rows of all files):", total_rows_all)
print("Total unique image_id across all files:", len(total_unique_ids_all))


Found 16 CSV files under /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag

=== Per-file summary ===


                                                 file split_guess   rows  \
0   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       ERROR  ERROR   
1   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test      0   
2   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
3   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
4   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
5   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   1787   
6   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   5000   
7   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train  11787   
8   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   6787   
9   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train     10   
10  /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   1787   
11  /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   5000   
12  /content


=== Coverage summary ===


   split  total_gt  covered  missing
0  train     11787    11787        0
1    val      1474     1474        0
2   test      3316     3316        0


=== Duplicates across CSVs (same split) ===
Train duplicates across files: 11787
Val   duplicates across files: 1474
Test  duplicates across files: 3316

Saved missing lists to:
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_train_image_ids.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_val_image_ids.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_test_image_ids.csv

=== Totals across ALL caption CSVs (not de-duplicated by split) ===
Total rows (sum of rows of all files): 46528
Total unique image_id across all files: 16577


In [ ]:
# === Audit all caption CSV files under OUT_DIR & check coverage ===
from pathlib import Path
import pandas as pd
import re
from collections import defaultdict, Counter

# 1) Locate ALL CSVs recursively
out_dir = Path(OUT_DIR)
csv_files = sorted([p for p in out_dir.rglob("*.csv") if p.is_file()])

if not csv_files:
    print("No CSVs found under:", out_dir)
else:
    print(f"Found {len(csv_files)} CSV files under {out_dir}")

# 2) Helper to normalize columns in any CSV
def load_csv_norm(path: Path):
    df = pd.read_csv(path, low_memory=False)
    # normalize column names
    df.columns = [c.strip().lower() for c in df.columns]
    # common renames
    rename = {
        "image id": "image_id",
        "img_id": "image_id",
        "labels": "label"
    }
    for k, v in rename.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})
    # keep a safe subset
    for need in ["image_id", "label", "caption", "status"]:
        if need not in df.columns:
            # create if missing to simplify later code
            df[need] = pd.NA
    # coerce to str where needed
    df["image_id"] = df["image_id"].astype(str)
    df["label"] = df["label"].astype(str)
    return df

# 3) Guess split from filename (fallback: unknown)
def guess_split_from_name(name: str):
    n = name.lower()
    if "train" in n: return "train"
    if re.search(r"\bval\b|valid", n): return "val"
    if "test" in n: return "test"
    return "unknown"

# 4) Collect per-file stats
rows = []
per_split_covered_ids = defaultdict(set)
global_seen_ids = set()

for p in csv_files:
    try:
        df = load_csv_norm(p)
    except Exception as e:
        rows.append({
            "file": str(p),
            "split_guess": "ERROR",
            "rows": "ERROR",
            "unique_image_id": "ERROR",
            "ok": "ERROR",
            "missing_image": "ERROR",
            "text_only": "ERROR",
            "note": f"failed to read: {e}"
        })
        continue

    split_guess = guess_split_from_name(p.name)
    n_rows = len(df)
    n_unique = df["image_id"].nunique()

    # status counts if available
    ok_cnt = (df["status"] == "OK").sum() if "status" in df.columns else pd.NA
    miss_cnt = (df["status"] == "MISSING_IMAGE").sum() if "status" in df.columns else pd.NA
    text_only_cnt = (df["status"] == "TEXT_ONLY").sum() if "status" in df.columns else pd.NA

    rows.append({
        "file": str(p),
        "split_guess": split_guess,
        "rows": n_rows,
        "unique_image_id": n_unique,
        "ok": ok_cnt if pd.notna(ok_cnt) else "",
        "missing_image": miss_cnt if pd.notna(miss_cnt) else "",
        "text_only": text_only_cnt if pd.notna(text_only_cnt) else "",
        "note": ""
    })

    # accumulate coverage sets by split guess
    if split_guess in ("train", "val", "test"):
        per_split_covered_ids[split_guess].update(df["image_id"].dropna().astype(str).tolist())
    else:
        # unknown files: still add into the global set; might help coverage if ids are unique anyway
        global_seen_ids.update(df["image_id"].dropna().astype(str).tolist())

# 5) Show per-file summary table
summary_df = pd.DataFrame(rows).sort_values(["split_guess", "file"]).reset_index(drop=True)
print("\n=== Per-file summary ===")
display(summary_df)

# 6) Build ground-truth sets from your original splits (assumes train/val/test DataFrames exist)
train_ids = set(train["image_id"].astype(str))
val_ids   = set(val["image_id"].astype(str))
test_ids  = set(test["image_id"].astype(str))

# If there are "unknown" caption files, include them in coverage heuristically
# (they might contain train/val/test ids; intersection will still work)
covered_train = set(per_split_covered_ids["train"]) | (global_seen_ids & train_ids)
covered_val   = set(per_split_covered_ids["val"])   | (global_seen_ids & val_ids)
covered_test  = set(per_split_covered_ids["test"])  | (global_seen_ids & test_ids)

missing_train = sorted(train_ids - covered_train)
missing_val   = sorted(val_ids   - covered_val)
missing_test  = sorted(test_ids  - covered_test)

print("\n=== Coverage summary ===")
coverage_rows = [
    {"split": "train", "total_gt": len(train_ids), "covered": len(covered_train), "missing": len(missing_train)},
    {"split": "val",   "total_gt": len(val_ids),   "covered": len(covered_val),   "missing": len(missing_val)},
    {"split": "test",  "total_gt": len(test_ids),  "covered": len(covered_test),  "missing": len(missing_test)},
]
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

# 7) Duplicates check across ALL files (per split)
def find_dups_across_files(target_ids, files):
    id_counter = Counter()
    for p in files:
        try:
            df = load_csv_norm(p)
        except Exception:
            continue
        ids = set(df["image_id"].astype(str))
        id_counter.update(ids & target_ids)
    dups = [i for i, c in id_counter.items() if c > 1]
    return dups

train_dups = find_dups_across_files(train_ids, csv_files)
val_dups   = find_dups_across_files(val_ids, csv_files)
test_dups  = find_dups_across_files(test_ids, csv_files)

print("\n=== Duplicates across CSVs (same split) ===")
print(f"Train duplicates across files: {len(train_dups)}")
print(f"Val   duplicates across files: {len(val_dups)}")
print(f"Test  duplicates across files: {len(test_dups)}")

# 8) (Optional) Save missing IDs to OUT_DIR
miss_train_path = out_dir / "audit_missing_train_image_ids.csv"
miss_val_path   = out_dir / "audit_missing_val_image_ids.csv"
miss_test_path  = out_dir / "audit_missing_test_image_ids.csv"

pd.DataFrame({"image_id": missing_train}).to_csv(miss_train_path, index=False)
pd.DataFrame({"image_id": missing_val}).to_csv(miss_val_path, index=False)
pd.DataFrame({"image_id": missing_test}).to_csv(miss_test_path, index=False)

print("\nSaved missing lists to:")
print(" -", miss_train_path)
print(" -", miss_val_path)
print(" -", miss_test_path)

# 9) Totals across ALL caption CSVs (for sanity)
total_rows_all = sum(r["rows"] for r in rows if isinstance(r["rows"], (int, float)))
total_unique_ids_all = set()
for p in csv_files:
    try:
        df = load_csv_norm(p)
        total_unique_ids_all.update(df["image_id"].astype(str))
    except Exception:
        pass

print("\n=== Totals across ALL caption CSVs (not de-duplicated by split) ===")
print("Total rows (sum of rows of all files):", total_rows_all)
print("Total unique image_id across all files:", len(total_unique_ids_all))



Found 17 CSV files under /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag

=== Per-file summary ===


                                                 file split_guess   rows  \
0   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       ERROR  ERROR   
1   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test      0   
2   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
3   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
4   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...        test   3316   
5   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   1787   
6   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   5000   
7   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train  11787   
8   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train      0   
9   /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train     10   
10  /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   1787   
11  /content/drive/MyDrive/Skin_Concepts/Skin_Conc...       train   5000   
12  /content


=== Coverage summary ===


   split  total_gt  covered  missing
0  train     11787    11787        0
1    val      1474     1474        0
2   test      3316     3316        0


=== Duplicates across CSVs (same split) ===
Train duplicates across files: 11787
Val   duplicates across files: 1474
Test  duplicates across files: 3316

Saved missing lists to:
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_train_image_ids.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_val_image_ids.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/audit_missing_test_image_ids.csv

=== Totals across ALL caption CSVs (not de-duplicated by split) ===
Total rows (sum of rows of all files): 46318
Total unique image_id across all files: 16577


In [ ]:
from pathlib import Path

OUT_DIR = CAPTIONS_DIR

all_csvs = sorted(Path(OUT_DIR).glob("*.csv"))
print(f"Found {len(all_csvs)} CSVs:")
for f in all_csvs:
    print("-", f.name)


Found 12 CSVs:
- againnewcaptions_train_10kto20k_qwen_rag_visual_only.csv
- againnewcaptions_train_5kto10k_qwen_rag_visual_only.csv
- againnewcaptions_train_qwen_rag_visual_only.csv
- anewcaptions_val_qwen_rag_visual_only.csv
- audit_missing_test_image_ids.csv
- audit_missing_train_image_ids.csv
- audit_missing_val_image_ids.csv
- captions_train_sample10_visual_only.csv
- combined_captions_all.csv
- newcaptions_train_10kto20k_minimal.csv
- newcaptions_train_first5k_qwen_rag_visual_only.csv
- nnewcaptions_test_qwen_rag_visual_only.csv


In [ ]:
from pathlib import Path
import pandas as pd

# === Base output folder ===
OUT_DIR = CAPTIONS_DIR

# === Explicitly define the 3 train CSVs ===
train_parts = [
    f"{OUT_DIR}/againnewcaptions_train_qwen_rag_visual_only.csv",
    f"{OUT_DIR}/againnewcaptions_train_10kto20k_qwen_rag_visual_only.csv",
    f"{OUT_DIR}/againnewcaptions_train_5kto10k_qwen_rag_visual_only.csv",
    f"{OUT_DIR}/newcaptions_train_first5k_qwen_rag_visual_only.csv",
]

# === Validation and Test CSVs ===
val_csv  = f"{CAPTIONS_DIR}/fresh_run_20251021_174013/captions_val_qwen_rag_visual_only.csv"
test_csv = f"{CAPTIONS_DIR}/fresh_run_20251021_174013/captions_test_qwen_rag_visual_only.csv"

print("🧩 Using train CSVs:")
for f in train_parts:
    print(" -", f)

# === Helper to clean CSVs ===
def load_clean_csv(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip().lower() for c in df.columns]
    rename_map = {"img_id": "image_id", "image id": "image_id", "labels": "label"}
    for k, v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})
    keep = [c for c in ["image_id","label","caption"] if c in df.columns]
    df = df[keep].dropna(subset=["image_id"])
    df["image_id"] = df["image_id"].astype(str)
    return df

# === Load each split ===
train_dfs = [load_clean_csv(p) for p in train_parts]
train_df = pd.concat(train_dfs, ignore_index=True)
train_df["split"] = "train"

val_df = load_clean_csv(val_csv)
val_df["split"] = "val"

test_df = load_clean_csv(test_csv)
test_df["split"] = "test"

# === Combine all ===
combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["image_id"], keep="first")

print("\nRows per split:")
print(combined_df["split"].value_counts())
print("Total combined rows:", len(combined_df))

# === Save final CSV ===
final_path = f"{OUT_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv"
combined_df.to_csv(final_path, index=False)
print("\n✅ Saved combined CSV:", final_path)

# === Preview sample ===
display(combined_df.sample(5))


🧩 Using train CSVs:
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_train_qwen_rag_visual_only.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_train_10kto20k_qwen_rag_visual_only.csv
 - /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/againnewcaptions_train_5kto10k_qwen_rag_visual_only.csv

Rows per split:
split
train    11787
test      3316
val       1474
Name: count, dtype: int64
Total combined rows: 16577

✅ Saved combined CSV: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv


                               image_id                               label  \
6173   b18b6e010b16dcee28deb5160b13b1e1                basal cell carcinoma   
8225   af12b74a13cba6c604aa3ee5fe9e20e2                    sun damaged skin   
21814  a4b48f03133497b7ff25ec2af7505ef1  superficial spreading melanoma ssm   
21532  987cbb71cee093391ec0b8b8ee241f1d       erythema annulare centrifigum   
18831  ed92739e1e824ff494bbdffdf68a5e25                       aplasia cutis   

                                                 caption  split  
6173                                                 NaN  train  
8225                                                 NaN  train  
21814  The image shows a superficial spreading melano...   test  
21532  The image shows a circular;  arciform;  and po...   test  
18831  The image shows a scalp defect;  which is a co...    val  

In [ ]:
import pandas as pd

# Path to your combined file
csv_path = f"{CAPTIONS_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv"

# Load CSV
df = pd.read_csv(csv_path)

# Normalize column names
df.columns = [c.strip().lower() for c in df.columns]

# Count missing captions
total_rows = len(df)
missing_captions = df["caption"].isna().sum()
non_missing_captions = total_rows - missing_captions

print(f"🧾 Total rows: {total_rows}")
print(f"❌ Missing captions: {missing_captions}")
print(f"✅ Non-missing captions: {non_missing_captions}")
print(f"📉 Percentage missing: {(missing_captions / total_rows) * 100:.2f}%")

# Optional: show some missing entries
missing_examples = df[df["caption"].isna()].head(10)
display(missing_examples)


🧾 Total rows: 16577
❌ Missing captions: 6866
✅ Non-missing captions: 9711
📉 Percentage missing: 41.42%


                              image_id                     label caption  \
4921  c43d73c0b5c34cb68d90545ee8bfc8f7              folliculitis     NaN   
4922  d8f122b99779ad61a1fda1ee1cadeb84               scleroderma     NaN   
4923  824a808cb69ba9b4e018d43fc73bce1c        granuloma annulare     NaN   
4924  b80457571db92b00ac22d66a309fb0c7      urticaria pigmentosa     NaN   
4925  f49e19687bcf242c59d1059799a30580  porokeratosis of mibelli     NaN   
4926  dcfba7fa975bf8013685b6ec6d979fb9         keratosis pilaris     NaN   
4927  cdbb50983b24266ce74b5d78b0d5589c         actinic keratosis     NaN   
4928  cb4b0d79ed51c820c5a12b1fc8e76580          pityriasis rosea     NaN   
4929  6dd80cd6cb04ab0d7418870bbce3c471           dariers disease     NaN   
4930  b3d4d798129927115ab5fba938f5453f           neurodermatitis     NaN   

      split  
4921  train  
4922  train  
4923  train  
4924  train  
4925  train  
4926  train  
4927  train  
4928  train  
4929  train  
4930  train  

In [ ]:
# ===== Caption ONLY the rows with missing captions, using your existing pipeline =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# (optional) keep-alive (same as your other cells)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write("."); sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Keep-alive on.")
except Exception:
    pass

# -------------------------------------------------------------------
# Paths
OUT_DIR = CAPTIONS_DIR  # adjust if needed
combined_csv = f"{OUT_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv"

# Patch output CSVs (resume-safe, will append if interrupted)
patch_train = f"{OUT_DIR}/patch_missing_train_qwen_rag_visual_only.csv"
patch_val   = f"{OUT_DIR}/patch_missing_val_qwen_rag_visual_only.csv"
patch_test  = f"{OUT_DIR}/patch_missing_test_qwen_rag_visual_only.csv"

# Clean 0-byte leftovers
for f in (patch_train, patch_val, patch_test):
    p = Path(f)
    if p.exists() and p.stat().st_size == 0:
        p.unlink()
        print("Removed empty file:", f)

# -------------------------------------------------------------------
# Load combined and find missing
df_all = pd.read_csv(combined_csv)
df_all.columns = [c.strip().lower() for c in df_all.columns]

if "split" not in df_all.columns:
    raise ValueError("Combined CSV must contain a 'split' column (train/val/test).")

missing = df_all[df_all["caption"].isna()].copy()
print(f"🔎 Found {len(missing)} rows with missing captions in combined CSV.")

# Split by split name
miss_train_ids = set(missing.loc[missing["split"]=="train", "image_id"].astype(str))
miss_val_ids   = set(missing.loc[missing["split"]=="val",   "image_id"].astype(str))
miss_test_ids  = set(missing.loc[missing["split"]=="test",  "image_id"].astype(str))

print(f"  • train missing: {len(miss_train_ids)}")
print(f"  • val   missing: {len(miss_val_ids)}")
print(f"  • test  missing: {len(miss_test_ids)}")

# -------------------------------------------------------------------
# Build the subsets from your ORIGINAL split DataFrames (train, val, test)
# (Assumes you still have the original DataFrames `train`, `val`, `test` in memory)
def subset_by_ids(df_split, ids):
    if not ids:
        return pd.DataFrame(columns=df_split.columns)
    df = df_split.copy()
    df["image_id"] = df["image_id"].astype(str)
    return df[df["image_id"].isin(ids)].reset_index(drop=True)

train_missing_df = subset_by_ids(train, miss_train_ids)
val_missing_df   = subset_by_ids(val,   miss_val_ids)
test_missing_df  = subset_by_ids(test,  miss_test_ids)

print(f"Subsets prepared -> train: {len(train_missing_df)}, val: {len(val_missing_df)}, test: {len(test_missing_df)}")

# -------------------------------------------------------------------
# Run your existing captioner ONLY on missing subsets
if len(train_missing_df):
    print(f"\n🟡 Captioning missing TRAIN rows: {len(train_missing_df)}")
    caption_split_resumable(train_missing_df, "train", patch_train)

if len(val_missing_df):
    print(f"\n🟡 Captioning missing VAL rows:   {len(val_missing_df)}")
    caption_split_resumable(val_missing_df, "val", patch_val)

if len(test_missing_df):
    print(f"\n🟡 Captioning missing TEST rows:  {len(test_missing_df)}")
    caption_split_resumable(test_missing_df, "test", patch_test)

# -------------------------------------------------------------------
# Merge patches back into the combined CSV
def load_patch(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame(columns=["image_id","caption","status"])
    dfp = pd.read_csv(p)
    dfp.columns = [c.strip().lower() for c in dfp.columns]
    # prefer OK, else TEXT_ONLY; ignore MISSING_IMAGE
    if "status" in dfp.columns:
        dfp = dfp[dfp["status"].isin(["OK","TEXT_ONLY"])]
    keep = [c for c in ["image_id","caption"] if c in dfp.columns]
    dfp = dfp[keep].dropna(subset=["image_id"]).copy()
    dfp["image_id"] = dfp["image_id"].astype(str)
    return dfp

patches = []
for p in (patch_train, patch_val, patch_test):
    patches.append(load_patch(p))
patch_df = pd.concat(patches, ignore_index=True) if patches else pd.DataFrame(columns=["image_id","caption"])

print(f"\n📦 Patch rows ready to merge: {len(patch_df)}")

# Index combined by image_id for fast fill
df_all["image_id"] = df_all["image_id"].astype(str)
patch_map = dict(zip(patch_df["image_id"], patch_df["caption"]))

# Fill only where caption is NaN and we have a patch
mask_missing = df_all["caption"].isna()
df_all.loc[mask_missing, "caption"] = df_all.loc[mask_missing, "image_id"].map(patch_map)

# Save a new filled file
filled_path = f"{CAPTIONS_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv"
df_all.to_csv(filled_path, index=False)
print(f"✅ Saved with filled captions:\n{filled_path}")

# Quick report
new_missing = df_all["caption"].isna().sum()
print(f"Missing captions BEFORE: {len(missing)} | AFTER: {new_missing}")


✅ Keep-alive on.
🔎 Found 6866 rows with missing captions in combined CSV.
  • train missing: 6866
  • val   missing: 0
  • test  missing: 0
Subsets prepared -> train: 6866, val: 0, test: 0

🟡 Captioning missing TRAIN rows: 6866
[train] Resuming — already has 6866 rows


Generating (train): 100%|██████████| 6866/6866 [00:00<00:00, 27788.72it/s]


[train] Wrote 0 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/patch_missing_train_qwen_rag_visual_only.csv

📦 Patch rows ready to merge: 514
✅ Saved with filled captions:
/content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv
Missing captions BEFORE: 6866 | AFTER: 6352


In [ ]:
# ===== Re-run missing caption generation from scratch =====
from pathlib import Path
import pandas as pd, os, sys, time, threading

# -------------------------------------------------------------------
# 🛡️ Keep-alive (Colab)
try:
    from google.colab import output
    output.eval_js("""
      (function(){
        if (window.__keepAliveInterval) return;
        window.__keepAliveInterval = setInterval(()=>console.log("↻ keepalive"), 60000);
      })();
    """)
    def _heartbeat():
        while True:
            time.sleep(60)
            sys.stdout.write(".")
            sys.stdout.flush()
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("✅ Keep-alive on.")
except Exception:
    pass

# -------------------------------------------------------------------
OUT_DIR = CAPTIONS_DIR
combined_csv = f"{OUT_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED.csv"
patch_train = f"{OUT_DIR}/patch_missing_train_qwen_rag_visual_only.csv"

# delete old patch file if exists (start fresh)
p = Path(patch_train)
if p.exists():
    p.unlink()
    print("🧹 Deleted old patch file to start fresh.")

# -------------------------------------------------------------------
df_all = pd.read_csv(combined_csv)
df_all.columns = [c.strip().lower() for c in df_all.columns]

missing = df_all[df_all["caption"].isna()].copy()
miss_train = missing[missing["split"] == "train"]
print(f"🔎 Missing captions: total {len(missing)}, train {len(miss_train)}")

# --- subset from your main train DataFrame ---
def subset_by_ids(df_split, ids):
    if not ids:
        return pd.DataFrame(columns=df_split.columns)
    df = df_split.copy()
    df["image_id"] = df["image_id"].astype(str)
    return df[df["image_id"].isin(ids)].reset_index(drop=True)

train_missing_df = subset_by_ids(train, miss_train["image_id"].astype(str).tolist())
print(f"🧠 Prepared subset with {len(train_missing_df)} missing images.")

# --- run captioning fresh ---
caption_split_resumable(train_missing_df, "train", patch_train)

# -------------------------------------------------------------------
# Merge back into combined CSV
patch_df = pd.read_csv(patch_train)
patch_df.columns = [c.strip().lower() for c in patch_df.columns]
patch_df = patch_df[patch_df["status"].isin(["OK","TEXT_ONLY"])]

patch_map = dict(zip(patch_df["image_id"].astype(str), patch_df["caption"]))
df_all["image_id"] = df_all["image_id"].astype(str)
mask_missing = df_all["caption"].isna()
df_all.loc[mask_missing, "caption"] = df_all.loc[mask_missing, "image_id"].map(patch_map)

filled_path = f"{OUT_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED_FILLED.csv"
df_all.to_csv(filled_path, index=False)

print(f"\n✅ Saved refreshed captions to:\n{filled_path}")
print("Still missing:", df_all["caption"].isna().sum())


✅ Keep-alive on.
🧹 Deleted old patch file to start fresh.
🔎 Missing captions: total 6352, train 6352
🧠 Prepared subset with 6352 missing images.
[train] No previous file → starting fresh.


Generating (train):   0%|          | 8/6352 [00:31<6:18:44,  3.58s/it]

.

Generating (train):   0%|          | 10/6352 [00:38<6:14:26,  3.54s/it]

.

...

.

Generating (train): 100%|█████████▉| 6343/6352 [6:09:59<00:31,  3.51s/it]

.

Generating (train): 100%|██████████| 6352/6352 [6:10:31<00:00,  3.50s/it]


[train] Wrote 6352 new rows → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/patch_missing_train_qwen_rag_visual_only.csv

✅ Saved refreshed captions to:
/content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/ALL_CAPTIONS_QWEN_RAG_COMBINED_FILLED.csv
Still missing: 0


In [ ]:
import pandas as pd
from pathlib import Path

# Path to your full combined file
OUT_DIR = CAPTIONS_DIR
combined_csv = f"{OUT_DIR}/ALL_CAPTIONS_QWEN_RAG_COMBINED_FILLED.csv"

# Load the combined CSV
df = pd.read_csv(combined_csv)
df.columns = [c.strip().lower() for c in df.columns]

# Check that the split column exists
if "split" not in df.columns:
    raise ValueError("❌ The CSV must contain a 'split' column (train/val/test).")

# Drop any duplicates
df = df.drop_duplicates(subset=["image_id", "split"], keep="last")

# Print counts for each split
print("📊 Split distribution:")
print(df["split"].value_counts())

# Create output paths
train_path = f"{OUT_DIR}/FINAL_TRAIN_QWEN_RAG_FILLED.csv"
val_path   = f"{OUT_DIR}/FINAL_VAL_QWEN_RAG_FILLED.csv"
test_path  = f"{OUT_DIR}/FINAL_TEST_QWEN_RAG_FILLED.csv"

# Save each subset
df[df["split"] == "train"].to_csv(train_path, index=False)
df[df["split"] == "val"].to_csv(val_path, index=False)
df[df["split"] == "test"].to_csv(test_path, index=False)

print("\n✅ Files saved:")
print(f"Train → {train_path}  ({len(df[df['split']=='train'])} rows)")
print(f"Val   → {val_path}    ({len(df[df['split']=='val'])} rows)")
print(f"Test  → {test_path}   ({len(df[df['split']=='test'])} rows)")


📊 Split distribution:
split
train    11787
test      3316
val       1474
Name: count, dtype: int64

✅ Files saved:
Train → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/FINAL_TRAIN_QWEN_RAG_FILLED.csv  (11787 rows)
Val   → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/FINAL_VAL_QWEN_RAG_FILLED.csv    (1474 rows)
Test  → /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/FINAL_TEST_QWEN_RAG_FILLED.csv   (3316 rows)
